In [ ]:
# =============================================================================
# CELL 1 / 8  —  SETUP, IMPORTS, BEST-HYPERPARAMETER CONFIG, PROMPTS, DATASET
# =============================================================================
# This notebook consolidates the whole domain-adaptive patch-mining pipeline
# into 8 cells. Every cell opens with a SECTION map so a specific fix is easy
# to locate. All corrections vs the previous version are tagged [v14 FIX ...]
# and summarised in FIXES.md.
#
#   SECTION 1.1  installs (commented — uncomment on a fresh runtime)
#   SECTION 1.2  imports
#   SECTION 1.3  device + reproducibility
#   SECTION 1.4  run-scope + BEST-HYPERPARAMETER config (baked defaults)
#   SECTION 1.5  scoring knobs, thresholds, patch geometry, persistence
#   SECTION 1.6  zero-shot prompts  (POSITIVE + NEW NEGATIVE pairs  [v14 FIX A])
#   SECTION 1.7  dataset-specific paths (NIH / MIMIC)
#   SECTION 1.8  counters + small helpers
# -----------------------------------------------------------------------------

# ============================ SECTION 1.1 — installs =========================
# Uncomment on a fresh Kaggle/Colab runtime:
# !pip install -q open_clip_torch ftfy regex tqdm
# !pip install -q git+https://github.com/openai/CLIP.git
# !pip install -q "transformers==5.0.0"

# ============================ SECTION 1.2 — imports ==========================
import os, re, gc, sys, copy, math, time, glob, json, random, pickle, warnings
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import clip
import torchvision.transforms as T
from PIL import Image, ImageDraw

from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.ndimage import label as ndimage_label, binary_dilation
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="torch")

# ==================== SECTION 1.3 — device + reproducibility =================
_T_START = time.time()
USE_CUDA = torch.cuda.is_available()
Device = torch.device("cuda" if USE_CUDA else "cpu")

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if USE_CUDA:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.set_float32_matmul_precision("high")

TARGET_CLASSES = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
N_CLASSES = len(TARGET_CLASSES)

# ============ SECTION 1.4 — run-scope + BEST-HYPERPARAMETER config ===========
# Defaults below are the "best known-good" settings from the v13 ablation plus
# the v14 corrections. Flip SMOKE_TEST / FAST_MODE only for a quick dry-run.
SMOKE_TEST             = False   # tiny real-data dry run (needs data + ckpt)
SMOKE_TRAIN_N          = 40
SMOKE_VAL_N            = 20
MULTICLASS_ONLY        = True
FAST_MODE              = False   # [v14 FIX C] keep False: FAST_MODE drops the 64px
                                 # scale that Atelectasis (small/subtle) needs.
MAX_TRAIN_IMAGES       = None
MAX_VAL_IMAGES         = 2000
USE_LABEL_BACKUP_TRAIN = False
USE_LABEL_BACKUP_VAL   = False

WALLCLOCK_BUDGET_HOURS = 9.0
CKPT_EVERY_IMAGES      = 500
CKPT_EVERY_MIN         = 12
VIS_MAX_PER_SPLIT      = 150

IMAGE_SIZE             = 512
CLIP_EMBED_DIM         = 512
PATCH_ENCODE_BATCH_SZ  = 128
TEXT_ENCODE_BATCH_SZ   = 64
ENABLE_GRADCAM         = True
ENABLE_ZOOM_REFINEMENT = True
ENABLE_VISUALIZATION   = False

# ---- [v14 FIX A/B] scoring-model switches (the two biggest catastrophic-output causes)
USE_BINARY_PROMPTS  = True   # [v14 FIX A] class prob = softmax over a POS/NEG prompt
                             # PAIR per class (CheXzero method), NOT a 5-way softmax
                             # across the mutually-competing target classes. The old
                             # 5-way softmax made co-occurring findings suppress each
                             # other's probability — the primary reason per-class
                             # AUROC sat at chance.
GLOBAL_ENSEMBLE_ALPHA = 0.50 # [v14 FIX B] final class score =
                             #   alpha * whole-image CheXzero prob
                             # + (1-alpha) * patch-max prob.
                             # alpha=1.0 => pure global baseline (report it!);
                             # alpha=0.0 => pure patch-mining (old behaviour).

MAX_ITER                 = 2 if FAST_MODE else 3
MAX_CANDIDATE_BOXES      = 100 if FAST_MODE else 160
MAX_ZOOM_CANDIDATE_BOXES = 80  if FAST_MODE else 120
MIN_BOX_SIZE = 24
DISCARD_UNCERTAIN = True

# ============ SECTION 1.5 — scoring weights, thresholds, geometry ============
SCORE_W_SEM  = 0.45
SCORE_W_PROB = 0.35
SCORE_W_GC   = 0.20

SEMANTIC_THRESHOLD = 0.16
SEMANTIC_THRESHOLD_PER_CLASS = {
    "Atelectasis": 0.16, "Cardiomegaly": 0.18, "Consolidation": 0.14,
    "Edema": 0.15, "Pleural Effusion": 0.18,
}
GRADCAM_WEAK_THR   = 0.08
SPATIAL_THRESH     = 0.18
CONF_THRESHOLD     = 0.30 if FAST_MODE else 0.40
FALLBACK_TOP_K     = 2
# [v13 FIX F6] Master switch for the causal-gate fallback. The v13 ablation showed
# G:no_fallback is the ONLY statistically significant arm and that the Edema
# headline is fallback-driven — so this is exposed prominently. Keep True to match
# the reported AUROC; set False for the honest "gate may abstain" causal claim.
ALLOW_FALLBACK     = True
MIN_CAUSAL_PATCHES = 2
TOP_K_PER_FIND     = 3 if FAST_MODE else 5
STRIDE_BASE        = 32
# [v14 FIX C] full multi-scale set — 64px kept for small findings.
PATCH_SCALES       = [128, 256] if FAST_MODE else [64, 128, 256]
ZOOM_SCALES        = [64] if FAST_MODE else [48, 64, 96]
GRADCAM_SNIPPET_W  = 0.65
GRADCAM_CLASS_W    = 0.35

CAUSAL_CONTAIN_MODE    = "peak_or_cover"
CAUSAL_MASK_COVER_FRAC = 0.35
CAUSAL_MASK_DILATE_PX  = 8

SPUR_IN_MAX_KEEP   = TOP_K_PER_FIND
SPUR_IN_GC_FLOOR   = GRADCAM_WEAK_THR
CAUSAL_QUERY_CLASS_BLEND = 0.35

# [v14 FIX C] per-class scale map is honoured by discover_patches_for_plan even
# when FAST_MODE is off — verify small findings keep 64px.
_SCALE_MAP = {
    "Atelectasis": [64, 128], "Cardiomegaly": [128, 256],
    "Consolidation": [128, 256], "Edema": [64, 128],
    "Pleural Effusion": [128, 256],
}

PERSISTENCE_N_LEVELS = 32
PERSISTENCE_TOP_K    = 2
PERSISTENCE_MIN_AREA = 16
PERSISTENCE_ENABLED  = True

OUTSIDE_ANAT_SCALES     = [64, 96, 128]
OUTSIDE_ANAT_INV_THRESH = 0.55
OUTSIDE_ANAT_MAX_CAND   = 120
OUTSIDE_ANAT_SIM_CAP    = 0.28
OUTSIDE_ANAT_MAX_KEEP   = 6
ARTIFACT_BORDER_FRAC = 0.15
ARTIFACT_MEAN_LOW    = 0.12
ARTIFACT_STD_HIGH    = 0.15
ARTIFACT_SCALES      = [64, 96]
ARTIFACT_SIM_CAP     = 0.28
ARTIFACT_MAX_KEEP    = 4

# ============ SECTION 1.6 — zero-shot prompts (POS + NEG pairs) ==============
# POSITIVE prompts (used for the class-text-matrix / GradCAM class objective).
ZS_PROMPTS = {
    "Atelectasis": [
        "atelectasis on chest x-ray", "lung collapse on chest radiograph",
        "plate-like atelectasis in the lung", "subsegmental atelectasis chest x-ray"],
    "Cardiomegaly": [
        "cardiomegaly on chest x-ray", "enlarged cardiac silhouette radiograph",
        "cardiac enlargement chest radiograph", "increased cardiothoracic ratio on x-ray"],
    "Consolidation": [
        "consolidation on chest x-ray", "airspace opacity in the lung",
        "lobar consolidation on radiograph", "air bronchogram in consolidated lung"],
    "Edema": [
        "pulmonary edema on chest x-ray", "bilateral interstitial edema radiograph",
        "vascular congestion in both lungs", "perihilar edema on chest radiograph"],
    "Pleural Effusion": [
        "pleural effusion on chest x-ray", "blunting of costophrenic angle",
        "pleural fluid on chest radiograph", "layering pleural effusion x-ray"],
}
# [v14 FIX A] NEGATIVE prompts — the second half of each CheXzero binary pair.
# Class probability becomes softmax([sim_pos, sim_neg]) so each class is scored
# independently of the other four (multi-label safe).
ZS_PROMPTS_NEG = {
    "Atelectasis": [
        "no atelectasis on chest x-ray", "no lung collapse",
        "lungs are fully expanded", "no volume loss in the lung"],
    "Cardiomegaly": [
        "normal heart size on chest x-ray", "no cardiomegaly",
        "normal cardiac silhouette", "normal cardiothoracic ratio"],
    "Consolidation": [
        "no consolidation on chest x-ray", "clear lungs without airspace opacity",
        "no lobar consolidation", "no air bronchogram"],
    "Edema": [
        "no pulmonary edema on chest x-ray", "no interstitial edema",
        "no vascular congestion", "clear lungs without edema"],
    "Pleural Effusion": [
        "no pleural effusion on chest x-ray", "sharp costophrenic angles",
        "no pleural fluid", "no layering effusion"],
}

# ============ SECTION 1.7 — dataset-specific configuration ===================
# >>> EDIT ME <<<
DATASET = "NIH"   # "NIH" or "MIMIC"
if DATASET == "NIH":
    VERSION = "v14"
    DATASET_NAME = "NIH-ChestXray"
    CSV_DIR       = "/kaggle/input/combinedreportimgpath"
    TRAIN_CSV     = os.path.join(CSV_DIR, "train_pairs_labeled.txt")
    VAL_CSV       = os.path.join(CSV_DIR, "val_pairs_labeled.txt")
    TEST_CSV      = os.path.join(CSV_DIR, "test_pairs_labeled.txt")
    CHEXZERO_CKPT = ("/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/"
                     "best_64_5e-05_original_22000_0.864.pt")
    NIH_IMAGE_DIR = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
    REPORT_DIR       = "/kaggle/input/datasets/anikazarin/nih-reports/reports"
    TRAIN_REPORT_DIR = os.path.join(REPORT_DIR, "train")
    VAL_REPORT_DIR   = os.path.join(REPORT_DIR, "val")
    TEST_REPORT_DIR  = os.path.join(REPORT_DIR, "test")
    OUT_DIR = "/kaggle/working/output"
    # [v14 FIX D] NIH ground-truth localization boxes for the A* patch metrics.
    BBOX_CSV = "/kaggle/input/datasets/organizations/nih-chest-xrays/data/BBox_List_2017.csv"
    WALLCLOCK_BUDGET_HOURS = 9.0
    CKPT_EVERY_IMAGES = 500; CKPT_EVERY_MIN = 12
    VIEW_FROM_REPORT = False; COMBINED_CSV = None; VAL_CARVE_FRAC = 0.08
else:
    VERSION = "v14_mimic"
    DATASET_NAME = "MIMIC-CXR"
    COMBINED_CSV  = "/kaggle/input/datasets/anikataf/mimic-cxr/mimic_cxr_combined_full.csv"
    CHEXZERO_CKPT = ("/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/"
                     "best_64_5e-05_original_22000_0.864.pt")
    TRAIN_CSV = VAL_CSV = TEST_CSV = None
    REPORT_DIR = TRAIN_REPORT_DIR = VAL_REPORT_DIR = TEST_REPORT_DIR = None
    OUT_DIR = "/kaggle/working"
    BBOX_CSV = None   # MIMIC ships no bounding boxes
    WALLCLOCK_BUDGET_HOURS = 9.5
    CKPT_EVERY_IMAGES = 400; CKPT_EVERY_MIN = 10
    VIEW_FROM_REPORT = True; VAL_CARVE_FRAC = 0.08

VIS_DIR = f"{OUT_DIR}/vis"
os.makedirs(OUT_DIR, exist_ok=True)
for _sp in ("train", "val"):
    os.makedirs(f"{VIS_DIR}/{_sp}", exist_ok=True)
STAGE2_OUT       = f"{OUT_DIR}/stage2_outputs.pkl"
STAGE3_RESULTS   = f"{OUT_DIR}/stage3_outputs.pkl"
STAGE3_CAUSAL_PT = f"{OUT_DIR}/stage3_causal.pt"
STAGE3_SPIN_PT   = f"{OUT_DIR}/stage3_spur_in.pt"
STAGE3_SPOUT_PT  = f"{OUT_DIR}/stage3_spur_out.pt"
STAGE3_META      = f"{OUT_DIR}/stage3_meta.pkl"
_DEADLINE = _T_START + WALLCLOCK_BUDGET_HOURS * 3600.0

# ============ SECTION 1.8 — counters + helpers ==============================
_COUNTERS = defaultdict(int)

def _budget_left_h(deadline=None):
    dl = deadline if deadline is not None else _DEADLINE
    return max(0.0, (dl - time.time()) / 3600.0)

def _atomic_pickle(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    os.replace(tmp, path)

print(f"[cell 1] DATASET={DATASET_NAME} ({VERSION}) | device={Device} | "
      f"binary_prompts={USE_BINARY_PROMPTS} | ensemble_alpha={GLOBAL_ENSEMBLE_ALPHA} | "
      f"fallback={ALLOW_FALLBACK} | fast={FAST_MODE}")


In [ ]:
# =============================================================================
# CELL 2 / 8  —  DATA STRUCTURES, ANATOMY PRIORS, REPORT PARSING
# =============================================================================
#   SECTION 2.1  dataclasses (QueryPlan / PatchDocument / ImagePatchResult)
#   SECTION 2.2  anatomy coordinates + GPU Gaussian heatmaps
#   SECTION 2.3  report parsing (sections, negation/uncertainty, view detect)
# These blocks were already correct in v13 and are carried over unchanged.
# -----------------------------------------------------------------------------

# ==================== SECTION 2.1 — data structures =========================
@dataclass
class AnatomicalPrior:
    region_name: str
    center_x: float; center_y: float; sigma_x: float; sigma_y: float

@dataclass
class PathologyQueryItem:
    pathology: str
    text_snippet: str
    query_vector: Optional[torch.Tensor] = None
    anatomical_prior: Optional[AnatomicalPrior] = None
    confidence: float = 1.0
    negated: bool = False
    source: str = "rule"

@dataclass
class QueryPlan:
    image_name: str; image_path: str; report_path: str
    view_position: str; findings_text: str; impression_text: str
    query_items: List[PathologyQueryItem] = field(default_factory=list)
    suppressed_regions: List[str] = field(default_factory=list)

@dataclass
class PatchDocument:
    image_name: str; pathology: str; scale: int
    box: Tuple[int, int, int, int]
    visual_embedding: torch.Tensor
    semantic_score: float; zeroshot_prob: float; gradcam_score: float
    combined_score: float; causal: bool; anatomical_region: str
    confidence: float; text_snippet: str
    zoom_level: int = 1
    spurious_source: str = "in_anatomy"
    selection_source: str = "threshold"

@dataclass
class ImagePatchResult:
    image_name: str; image_path: str; split: str
    causal_patches: List[PatchDocument] = field(default_factory=list)
    spurious_patches: List[PatchDocument] = field(default_factory=list)
    refined: bool = False; n_iterations: int = 1; used_fallback: bool = False

# ==================== SECTION 2.2 — anatomy + heatmaps ======================
ANATOMY_COORDS_PA = {
    "right lung": (0.28, 0.42, 0.14, 0.22), "left lung": (0.72, 0.42, 0.14, 0.22),
    "bilateral lungs": (0.50, 0.42, 0.38, 0.22), "lungs": (0.50, 0.42, 0.38, 0.22),
    "lung": (0.50, 0.42, 0.38, 0.22),
    "right upper lobe": (0.28, 0.18, 0.10, 0.10), "right middle lobe": (0.28, 0.38, 0.10, 0.09),
    "right lower lobe": (0.28, 0.60, 0.10, 0.11), "left upper lobe": (0.72, 0.18, 0.10, 0.10),
    "left lower lobe": (0.72, 0.60, 0.10, 0.11), "lower lobes": (0.50, 0.62, 0.32, 0.10),
    "upper lobes": (0.50, 0.16, 0.32, 0.09), "lower lobe": (0.50, 0.62, 0.32, 0.10),
    "right costophrenic angle": (0.22, 0.81, 0.07, 0.06),
    "left costophrenic angle": (0.78, 0.81, 0.07, 0.06),
    "costophrenic angles": (0.50, 0.81, 0.36, 0.06), "costophrenic angle": (0.50, 0.81, 0.36, 0.06),
    "costophrenic": (0.50, 0.81, 0.36, 0.06),
    "cardiac silhouette": (0.53, 0.48, 0.12, 0.14), "heart": (0.53, 0.48, 0.12, 0.14),
    "cardiomegaly": (0.53, 0.48, 0.12, 0.14), "mediastinum": (0.50, 0.34, 0.08, 0.18),
    "mediastinal": (0.50, 0.34, 0.08, 0.18), "hila": (0.50, 0.42, 0.08, 0.08),
    "right hilum": (0.39, 0.42, 0.05, 0.06), "left hilum": (0.60, 0.42, 0.05, 0.06),
    "hilar": (0.50, 0.42, 0.08, 0.08), "apices": (0.50, 0.07, 0.26, 0.05),
    "right apex": (0.28, 0.06, 0.08, 0.04), "left apex": (0.72, 0.06, 0.08, 0.04),
    "apex": (0.50, 0.07, 0.26, 0.05), "right pleura": (0.17, 0.44, 0.05, 0.26),
    "left pleura": (0.83, 0.44, 0.05, 0.26), "pleural space": (0.50, 0.44, 0.42, 0.26),
    "pleural": (0.50, 0.44, 0.42, 0.26), "right hemidiaphragm": (0.30, 0.76, 0.12, 0.04),
    "left hemidiaphragm": (0.70, 0.78, 0.12, 0.04), "diaphragm": (0.50, 0.77, 0.32, 0.04),
    "trachea": (0.50, 0.20, 0.04, 0.08), "carina": (0.50, 0.30, 0.04, 0.03),
    "ribs": (0.50, 0.45, 0.40, 0.26), "clavicles": (0.50, 0.09, 0.30, 0.03),
    "spine": (0.50, 0.45, 0.04, 0.30),
}
ANATOMY_COORDS_AP = {
    **ANATOMY_COORDS_PA,
    "cardiac silhouette": (0.53, 0.52, 0.16, 0.17), "heart": (0.53, 0.52, 0.16, 0.17),
    "cardiomegaly": (0.53, 0.52, 0.16, 0.17),
    "right upper lobe": (0.29, 0.20, 0.12, 0.12), "left upper lobe": (0.71, 0.20, 0.12, 0.12),
    "right lower lobe": (0.29, 0.62, 0.11, 0.12), "left lower lobe": (0.71, 0.62, 0.11, 0.12),
}
_ANATOMY_SORTED = sorted(ANATOMY_COORDS_PA.keys(), key=len, reverse=True)

def get_anatomy_coords(region, view="PA"):
    lut = ANATOMY_COORDS_AP if str(view).upper() == "AP" else ANATOMY_COORDS_PA
    key = str(region).lower().strip()
    if key in lut:
        return lut[key]
    for k in _ANATOMY_SORTED:
        if k in key:
            return lut[k]
    return (0.50, 0.42, 0.36, 0.24)

def make_gaussian_heatmap_gpu(cx, cy, sx, sy, H=IMAGE_SIZE, W=IMAGE_SIZE, device=None):
    if device is None:
        device = Device
    xs = torch.linspace(0, 1, W, device=device)
    ys = torch.linspace(0, 1, H, device=device)
    yg, xg = torch.meshgrid(ys, xs, indexing="ij")
    return torch.exp(-((xg - cx) ** 2 / (2 * sx ** 2) + (yg - cy) ** 2 / (2 * sy ** 2))).float()

def get_composite_heatmap_gpu(plan, H=IMAGE_SIZE, W=IMAGE_SIZE, device=None):
    if device is None:
        device = Device
    composite = torch.zeros(H, W, device=device)
    suppressed = torch.zeros(H, W, device=device)
    for item in plan.query_items:
        if item.negated or item.anatomical_prior is None:
            continue
        p = item.anatomical_prior
        composite += item.confidence * make_gaussian_heatmap_gpu(
            p.center_x, p.center_y, p.sigma_x, p.sigma_y, H, W, device)
    for region in plan.suppressed_regions:
        rx, ry, rsx, rsy = get_anatomy_coords(region, plan.view_position)
        suppressed += make_gaussian_heatmap_gpu(rx, ry, rsx, rsy, H, W, device)
    suppressed = suppressed.clamp(0, 1)
    maxc = composite.max()
    if maxc > 0:
        composite = composite / maxc
    return (composite * (1.0 - suppressed)).float()

# ==================== SECTION 2.3 — report parsing ==========================
_SECTION_HDR_PAT = re.compile(r"^\s*\d+\s*\)\s*", re.M)
_FINDINGS_PAT    = re.compile(r"^\s*(?:\d+\s*[\).:]\s*)?FINDINGS\b",   re.I | re.M)
_IMPRESSION_PAT  = re.compile(r"^\s*(?:\d+\s*[\).:]\s*)?IMPRESSION\b", re.I | re.M)
_NONCLINICAL_PAT = re.compile(
    r"^\s*(?:\d+\s*[\).:]\s*)?"
    r"(EXAM\s*/?\s*TECHNIQUES?|IMAGE\s*QUAL[A-Z]*\s*/?\s*TECHNICAL|"
    r"DEVICES\s*/?\s*LINES|LIMITATIONS?\s*/?\s*UNCERTAINTY|DOMAIN\s*VECTOR)", re.I | re.M)
_NEG_CUES = re.compile(
    r"\b(no\b|not\b|none\b|without|absent|clear\b|unremarkable|normal\b|"
    r"no evidence of|no acute|free of|negative for|unlikely|no significant|"
    r"no definite|is not seen|are not seen|resolved|ruled out)\b", re.I)
_UNC_CUES = re.compile(
    r"\b(possible|possibly|probable|probably|may\b|might|suspected|"
    r"cannot exclude|cannot rule out|question of|questionable|"
    r"differential|versus|vs\.?|borderline|likely|equivocal|indeterminate|"
    r"suggestive of|concerning for|could represent|may represent|"
    r"appears?\s+to\s+be|appears?\b|apparent|worrisome for|"
    r"further evaluation|correlate clinically)\b", re.I)
_VIS_LIMIT_CUES = re.compile(
    r"\bnot\s+(?:fully|well|completely|adequately|entirely)?\s*"
    r"(?:visualized|evaluated|assessed|included|imaged|seen on)", re.I)
_SCOPE_BREAK = re.compile(
    r"\b(but|however|although|though|except|aside from|other than|whereas|while)\b|[,;]", re.I)

_PROJECTION_FIELD = re.compile(
    r"projection\s*[:\(]\s*([A-Za-z]+(?:\s+(?:supine|erect|portable|upright))*)", re.I)
_PORTABLE_FIELD = re.compile(r"portable[-\s]?likelihood\s*[:\(]\s*(low|medium|moderate|high)", re.I)
_POSITION_FIELD = re.compile(r"patient\s+position\s*[:\(]\s*([A-Za-z\- ]+)", re.I)
_VIEW_AP_CUE = re.compile(r"\b(AP|SUPINE|SEMI[- ]?ERECT|SEMI[- ]?UPRIGHT|DECUBITUS)\b", re.I)
_VIEW_PA_CUE = re.compile(r"\bPA\b", re.I)

SEED_KEYWORDS = {
    "Atelectasis": ["atelectasis", "atelectatic", "volume loss", "plate-like", "discoid",
        "linear opacity", "linear opacities", "subsegmental", "crowding of",
        "elevated hemidiaphragm", "collapse"],
    "Cardiomegaly": ["cardiomegaly", "enlarged cardiac", "cardiac enlargement",
        "cardiothoracic ratio", "cardiac silhouette is enlarged", "heart size",
        "heart is enlarged", "heart appears enlarged", "enlarged heart",
        "cardiac silhouette", "cardiomediastinal silhouette", "cardiac contour"],
    "Consolidation": ["consolidation", "airspace opacity", "airspace opacities",
        "lobar opacity", "air bronchogram", "airspace disease", "patchy opacity",
        "patchy opacities", "focal opacity", "focal opacities", "increased opacity",
        "increased density", "ill-defined opacity", "ill-defined opacities",
        "infiltrate", "infiltrates", "dense opacity"],
    "Edema": ["edema", "oedema", "pulmonary congestion", "vascular congestion", "kerley",
        "perihilar", "interstitial markings", "vascular redistribution",
        "interstitial opacity", "interstitial opacities", "hazy opacity",
        "hazy opacities", "reticular opacity", "reticular opacities",
        "increased interstitial", "cephalization"],
    "Pleural Effusion": ["effusion", "pleural fluid", "blunting", "meniscus", "costophrenic",
        "pleural collection", "layering", "blunted costophrenic", "fluid in the pleural"],
}
DEFAULT_ANATOMY = {
    "Atelectasis": "lower lobes", "Cardiomegaly": "cardiac silhouette",
    "Consolidation": "right lower lobe", "Edema": "bilateral lungs",
    "Pleural Effusion": "right costophrenic angle",
}

def detect_view_from_report(raw, default="PA"):
    text = str(raw); head = text[:1200]
    m = _PROJECTION_FIELD.search(head)
    proj = m.group(1).strip().upper() if m else ""
    if proj.startswith("LATERAL"): return "LATERAL"
    if proj.startswith("AP"):      return "AP"
    if proj.startswith("PA"):      return "PA"
    mp = _PORTABLE_FIELD.search(head)
    if mp and mp.group(1).lower() in ("medium", "moderate", "high"): return "AP"
    mpos = _POSITION_FIELD.search(head)
    if mpos and re.search(r"supine|semi[- ]?erect|semi[- ]?upright|decubitus", mpos.group(1), re.I):
        return "AP"
    scan = re.sub(r"portable[-\s]?likelihood[^\n]*", " ", head, flags=re.I).upper()
    has_ap = bool(_VIEW_AP_CUE.search(scan)); has_pa = bool(_VIEW_PA_CUE.search(scan))
    if has_pa and not has_ap: return "PA"
    if has_ap and not has_pa: return "AP"
    return default

def parse_report_sections(raw):
    lines = str(raw).split("\n")
    sections = {"findings": "", "impression": ""}
    current, buf = None, []
    def _flush():
        if current and buf:
            prev = sections.get(current, ""); joined = "\n".join(buf).strip()
            sections[current] = (prev + "\n" + joined).strip() if prev else joined
    for line in lines:
        if _FINDINGS_PAT.search(line):     _flush(); current, buf = "findings", []
        elif _IMPRESSION_PAT.search(line): _flush(); current, buf = "impression", []
        elif _NONCLINICAL_PAT.search(line):_flush(); current, buf = None, []
        elif _SECTION_HDR_PAT.match(line): _flush(); current, buf = None, []
        elif current is not None:          buf.append(line)
    _flush()
    return sections

def _scoped(window, forward):
    m = _SCOPE_BREAK.search(window)
    if not m: return window
    return window[m.end():] if not forward else window[:m.start()]

def sentence_status(sentence, keyword):
    sl = str(sentence).lower(); kw = str(keyword).lower(); pos = sl.find(kw)
    if pos < 0: return "absent"
    prefix = _scoped(sl[:pos], forward=False)
    suffix = _scoped(sl[pos + len(kw):], forward=True)
    if _VIS_LIMIT_CUES.search(prefix) or _VIS_LIMIT_CUES.search(suffix): return "uncertain"
    if _NEG_CUES.search(prefix) or _NEG_CUES.search(suffix): return "negated"
    if _UNC_CUES.search(prefix) or _UNC_CUES.search(suffix): return "uncertain"
    return "positive"

def extract_keyword_sentences(text, keywords):
    ordered = sorted({str(k).lower() for k in keywords}, key=len, reverse=True)
    results = []
    for sent in re.split(r"[.\n;]", str(text)):
        sent = re.sub(r"\s+", " ", sent).strip()
        if len(sent) < 8: continue
        low = sent.lower()
        for kw in ordered:
            if kw in low:
                results.append((sent, sentence_status(sent, kw))); break
    return results

def extract_anatomy_mentions(text):
    tl = str(text).lower()
    return [k for k in _ANATOMY_SORTED if k in tl]

def _read_report(path):
    try:
        if isinstance(path, str) and os.path.exists(path):
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
    except Exception:
        _COUNTERS["reports_read_error"] += 1
    return ""

def _strip_nonclinical(raw):
    keep, drop = [], False
    for line in str(raw).split("\n"):
        if _FINDINGS_PAT.search(line) or _IMPRESSION_PAT.search(line):
            drop = False; continue
        if _NONCLINICAL_PAT.search(line) or _SECTION_HDR_PAT.match(line):
            drop = True; continue
        if not drop: keep.append(line)
    return "\n".join(keep).strip()

def _clinical_text_from_raw(raw):
    secs = parse_report_sections(raw)
    findings = secs.get("findings", ""); impression = secs.get("impression", "")
    clinical = (findings + "\n" + impression).strip()
    if len(clinical) < 20: clinical = _strip_nonclinical(raw)
    if len(clinical) < 20:
        _COUNTERS["reports_no_clinical_text"] += 1; clinical = str(raw)
    return findings, impression, clinical

print("[cell 2] data structures, anatomy priors, report parser ready.")


In [ ]:
# =============================================================================
# CELL 3 / 8  —  VOCAB MINING  +  STAGE-2 QUERY-PLAN BUILDER
# =============================================================================
#   SECTION 3.1  mine_vocab_from_reports (per-disease query phrases)
#   SECTION 3.2  label-column resolution + CSV reading
#   SECTION 3.3  rule-based query items + build_query_plan_for_row + run_stage2
# Carried over from v13 (report-parsing fixes already applied in cell 2).
# -----------------------------------------------------------------------------

# ==================== SECTION 3.1 — vocab mining ============================
def mine_vocab_from_reports(train_df, top_n=8, min_count=3, max_phrase_len=120):
    if "report_path" not in train_df.columns:
        print("report_path missing - seed vocab only.")
        return {d: {"query_phrases": [f"{d.lower()} on chest x-ray"],
                    "keywords": SEED_KEYWORDS[d], "default_anatomy": DEFAULT_ANATOMY[d]}
                for d in TARGET_CLASSES}
    report_paths = train_df["report_path"].dropna().astype(str).unique().tolist()
    print(f"\n{'='*70}\nVOCAB MINER: {len(report_paths):,} reports\n{'='*70}")
    disease_sentences = {d: [] for d in TARGET_CLASSES}; skipped = 0
    for path in tqdm(report_paths, desc="Mining vocab"):
        raw = _read_report(path)
        if not raw.strip():
            skipped += 1; continue
        _, _, clinical = _clinical_text_from_raw(raw)
        for disease in TARGET_CLASSES:
            for sent_raw in re.split(r"[.\n;]", clinical):
                sent = re.sub(r"\s+", " ", sent_raw.strip())
                kw = next((k for k in SEED_KEYWORDS[disease] if k.lower() in sent.lower()), None)
                if kw and sentence_status(sent, kw) == "positive" and 15 <= len(sent) <= max_phrase_len:
                    disease_sentences[disease].append(sent.lower())
    print(f"Skipped: {skipped:,}")
    mined = {}
    for d in TARGET_CLASSES:
        sents = disease_sentences[d]
        print(f"  {d:<20}: {len(sents):,}")
        if not sents:
            mined[d] = {"query_phrases": [f"{d.lower()} on chest x-ray"],
                        "keywords": SEED_KEYWORDS[d], "default_anatomy": DEFAULT_ANATOMY[d]}
            continue
        ctr = Counter(sents)
        top = [s for s, _ in sorted(
            [(s, c) for s, c in ctr.items() if c >= min_count and len(s) >= 20],
            key=lambda x: (-x[1], -len(x[0])))[:top_n]]
        if len(top) < 2:
            top = [s for s, _ in ctr.most_common(top_n)]
        mined[d] = {"query_phrases": top[:top_n], "keywords": SEED_KEYWORDS[d],
                    "default_anatomy": DEFAULT_ANATOMY[d]}
    return mined

# ==================== SECTION 3.2 — labels + CSV ============================
_LABEL_ALIASES = {
    "Atelectasis": ["Atelectasis"], "Cardiomegaly": ["Cardiomegaly"],
    "Consolidation": ["Consolidation"], "Edema": ["Edema"],
    "Pleural Effusion": ["Pleural Effusion", "Pleural_Effusion", "Effusion"],
}
def _detect_csv_sep(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        line = f.readline()
    return "\t" if "\t" in line and line.count("\t") >= line.count(",") else ","

def read_pairs_csv(path):
    return pd.read_csv(path, sep=_detect_csv_sep(path))

def _find_label_cols(df):
    cols = {c: c for c in df.columns}; lower = {c.lower(): c for c in df.columns}; found = {}
    for cls, aliases in _LABEL_ALIASES.items():
        hit = None
        for a in aliases:
            for cand in (a, a.replace(" ", "_"), a.lower(), a.replace(" ", "_").lower()):
                if cand in cols: hit = cols[cand]; break
                if cand in lower: hit = lower[cand]; break
            if hit: break
        if hit: found[cls] = hit
    return found

def filter_multiclass_rows(df, name):
    n0 = len(df); lcols = _find_label_cols(df)
    if lcols:
        mask = pd.Series(False, index=df.index)
        for _cls, col in lcols.items():
            v = pd.to_numeric(df[col], errors="coerce"); mask = mask | (v == 1.0)
        out = df[mask].reset_index(drop=True)
        print(f"  {name}: multi-class filter kept {len(out):,}/{n0:,} (label cols: {list(lcols.values())})")
        return out
    aliases = set()
    for al in _LABEL_ALIASES.values(): aliases |= set(al)
    def _has(s): return len(set(str(s).split("|")) & aliases) > 0
    for lc in ["Finding Labels", "finding_labels", "labels", "Finding_Labels", "Labels"]:
        if lc in df.columns:
            out = df[df[lc].apply(_has)].reset_index(drop=True)
            print(f"  {name}: multi-class filter via pipe column '{lc}' kept {len(out):,}/{n0:,}")
            return out
    print(f"  {name}: no label columns found - skipping filter.")
    return df

def _get_col(row, cands, default=""):
    for c in cands:
        if c in row.index: return row[c]
    return default

def _get_label_val(row, label):
    for c in [label, label.replace(" ", "_"), label.lower(), label.replace(" ", "_").lower()]:
        if c in row.index: return row[c]
    return None

def _resolve_image_name(row):
    name = str(_get_col(row, ["image_name", "Image Index", "image", "filename", "study_id"], ""))
    if not name or name.lower() == "nan":
        p = str(_get_col(row, ["image_path", "path"], ""))
        name = os.path.basename(p) if p else ""
    return name

# ==================== SECTION 3.3 — query plans ============================
def _rule_based_items(clinical, vocab, view, positive_labels, use_label_backup):
    disease_hits = {}
    for disease, entry in vocab.items():
        keywords = entry.get("keywords", SEED_KEYWORDS.get(disease, []))
        disease_hits[disease] = extract_keyword_sentences(clinical, keywords)
    suppressed = []
    for disease, hits in disease_hits.items():
        for sent, status in hits:
            if status == "negated":
                for region in extract_anatomy_mentions(sent):
                    if region not in suppressed: suppressed.append(region)
    report_detected = {d for d, hits in disease_hits.items()
                       if any(s == "positive" for _, s in hits)}
    candidates = set(report_detected)
    if use_label_backup: candidates = candidates | set(positive_labels)
    items = []
    for disease in sorted(candidates):
        if disease not in vocab: continue
        entry = vocab[disease]; hits = disease_hits.get(disease, [])
        pos_hits = [s for s, st in hits if st == "positive"]
        unc_hits = [s for s, st in hits if st == "uncertain"]
        if pos_hits:
            snippet = max(pos_hits, key=len)
            confidence = 1.00 if (use_label_backup and disease in positive_labels) else 0.80
            negated = False
        elif unc_hits and not DISCARD_UNCERTAIN:
            snippet = max(unc_hits, key=len); confidence = 0.55; negated = False
        elif disease in positive_labels and use_label_backup:
            snippet = entry.get("query_phrases", [f"{disease.lower()} on chest x-ray"])[0]
            confidence = 0.70; negated = False
        else:
            continue
        anatomy_hits = extract_anatomy_mentions(snippet)
        primary = anatomy_hits[0] if anatomy_hits else entry.get(
            "default_anatomy", DEFAULT_ANATOMY.get(disease, "lungs"))
        cx, cy, sx, sy = get_anatomy_coords(primary, view)
        items.append(PathologyQueryItem(
            pathology=disease, text_snippet=snippet, query_vector=None,
            anatomical_prior=AnatomicalPrior(primary, cx, cy, sx, sy),
            confidence=confidence, negated=negated, source="rule"))
    return items, suppressed

def build_query_plan_for_row(row, vocab, view, use_label_backup, view_from_report=False):
    img_name = _resolve_image_name(row)
    img_path = str(_get_col(row, ["image_path", "path"], ""))
    rpt_path = str(_get_col(row, ["report_path", "report", "text_path"], ""))
    raw = _read_report(rpt_path)
    if view not in ("PA", "AP"):
        view = detect_view_from_report(raw) if view_from_report else "PA"
    findings, impression, clinical = _clinical_text_from_raw(raw)
    plan = QueryPlan(image_name=img_name, image_path=img_path, report_path=rpt_path,
                     view_position=view, findings_text=findings, impression_text=impression)
    pos_labels = set()
    for d in TARGET_CLASSES:
        val = _get_label_val(row, d)
        try:
            if val is not None and not pd.isna(val) and float(val) == 1.0: pos_labels.add(d)
        except Exception:
            pass
    items, suppressed = _rule_based_items(clinical, vocab, view, pos_labels, use_label_backup)
    plan.query_items = items; plan.suppressed_regions = suppressed
    return plan

def run_stage2(df, vocab, max_imgs, desc, use_label_backup, view_default="PA", view_from_report=False):
    subset = df.iloc[:max_imgs] if max_imgs is not None else df
    plans = []
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc=desc):
        view = str(_get_col(row, ["View_Position", "View Position", "view_position", "view"],
                            view_default)).upper()
        if view not in ("PA", "AP"): view = "AUTO" if view_from_report else "PA"
        plans.append(build_query_plan_for_row(row, vocab, view, use_label_backup,
                                              view_from_report=view_from_report))
    return plans

def split_stats(plans, name):
    ni = sum(len(p.query_items) for p in plans)
    ne = sum(1 for p in plans if p.query_items)
    n_ap = sum(1 for p in plans if p.view_position == "AP")
    print(f"{name:<6}: {len(plans):,} | {ne:,} non-empty | {ni:,} items | "
          f"avg={ni/max(len(plans),1):.2f} | AP-view={n_ap:,}")

print("[cell 3] vocab miner + Stage-2 query-plan builder ready.")


In [ ]:
# =============================================================================
# CELL 4 / 8  —  CHEXZERO  +  BINARY PROMPTS  +  GRAD-CAM  +  SCORING
# =============================================================================
#   SECTION 4.1  load_chexzero (+ frozen variant)
#   SECTION 4.2  class-text matrix (5-way, GradCAM objective)  AND
#                [v14 FIX A] binary POS/NEG prompt matrices
#   SECTION 4.3  fill_query_vectors
#   SECTION 4.4  ViTGradCAM
#   SECTION 4.5  scoring: [v14 FIX A] decoupled binary class prob +
#                [v14 FIX B] whole-image global prob for the ensemble
# -----------------------------------------------------------------------------

# ==================== SECTION 4.1 — load CheXzero ==========================
def load_chexzero(ckpt_path, device):
    print("\n" + "=" * 70 + "\nLOADING CHEXZERO\n" + "=" * 70)
    assert os.path.exists(ckpt_path), f"Not found: {ckpt_path}"
    model, preprocess = clip.load("ViT-B/32", device=device, jit=False)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(state, dict):
        for k in ("state_dict", "model", "model_state_dict", "net"):
            if k in state: state = state[k]; break
    clean = {k.replace("module.", "").replace("model.", ""): v for k, v in state.items()}
    miss, unex = model.load_state_dict(clean, strict=False)
    print(f"Missing: {len(miss)} | Unexpected: {len(unex)}")
    print("CheXzero loaded." if len(miss) <= 50 else "Many missing keys.")
    model.eval().to(device)
    for p in model.parameters(): p.requires_grad_(False)
    for p in model.visual.transformer.resblocks[-1].parameters(): p.requires_grad_(True)
    return model, preprocess

def load_chexzero_model(ckpt_path, device):
    model, preprocess = clip.load("ViT-B/32", device=device, jit=False)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(state, dict):
        for k in ("state_dict", "model", "model_state_dict", "net"):
            if k in state: state = state[k]; break
    clean = {k.replace("module.", "").replace("model.", ""): v for k, v in state.items()}
    model.load_state_dict(clean, strict=False)
    model.eval().to(device)
    for p in model.parameters(): p.requires_grad_(False)
    return model, preprocess

# ==================== SECTION 4.2 — text matrices ==========================
@torch.no_grad()
def _encode_prompt_group(model, prompts, device):
    """Mean L2-normalised text embedding for a list of prompts -> [D]."""
    toks = clip.tokenize(prompts, truncate=True).to(device)
    e = model.encode_text(toks).float()
    e = e / e.norm(dim=-1, keepdim=True)
    m = e.mean(0); m = m / m.norm()
    return m

@torch.no_grad()
def build_class_text_matrix(model, device):
    """5-way positive-only CTM. Retained for the GradCAM class objective."""
    rows = [_encode_prompt_group(model, ZS_PROMPTS[c], device) for c in TARGET_CLASSES]
    ctm = torch.stack(rows).float()
    ls = model.logit_scale.exp().detach().float()
    print(f"CTM: {tuple(ctm.shape)} | logit_scale={ls.item():.2f}")
    return ctm, ls

@torch.no_grad()
def build_binary_prompt_matrices(model, device):
    """[v14 FIX A] Per-class POS and NEG prompt matrices -> (pos[C,D], neg[C,D]).
    Class probability is later softmax over each class's own {pos, neg} pair, so
    a finding is scored independently of the other four target classes."""
    pos = torch.stack([_encode_prompt_group(model, ZS_PROMPTS[c], device) for c in TARGET_CLASSES])
    neg = torch.stack([_encode_prompt_group(model, ZS_PROMPTS_NEG[c], device) for c in TARGET_CLASSES])
    print(f"Binary prompt matrices: pos{tuple(pos.shape)} neg{tuple(neg.shape)}")
    return pos.float(), neg.float()

# ==================== SECTION 4.3 — fill query vectors =====================
@torch.no_grad()
def fill_query_vectors(plans, model, device, ctm=None, bs=TEXT_ENCODE_BATCH_SZ):
    refs, texts = [], []
    for p in plans:
        for qi in p.query_items:
            refs.append(qi); texts.append(str(qi.text_snippet).strip() or qi.pathology)
    if not texts: return plans
    model.eval(); all_e = []
    for i in tqdm(range(0, len(texts), bs), desc="Text encode"):
        toks = clip.tokenize(texts[i:i + bs], truncate=True).to(device)
        e = model.encode_text(toks).float(); e = e / e.norm(dim=-1, keepdim=True)
        all_e.append(e.detach().cpu())
    all_e = torch.cat(all_e)
    ctm_cpu = ctm.detach().cpu().float() if ctm is not None else None
    a = float(CAUSAL_QUERY_CLASS_BLEND); n_blend = 0
    for qi, e in zip(refs, all_e):
        e = e.float()
        if ctm_cpu is not None and a > 0.0 and qi.pathology in TARGET_CLASSES:
            cprompt = ctm_cpu[TARGET_CLASSES.index(qi.pathology)]
            blended = (1.0 - a) * e + a * cprompt; nrm = blended.norm()
            if nrm > 0: e = (blended / nrm).float(); n_blend += 1
        qi.query_vector = e.float()
    if ctm_cpu is not None and a > 0.0:
        print(f"  causal query blend: {n_blend:,} query vectors enriched (weight={a:.2f})")
    return plans

# ==================== SECTION 4.4 — ViT Grad-CAM ==========================
class ViTGradCAM:
    def __init__(self, model, preprocess, device, ctm, logit_scale):
        self.model = model; self.preprocess = preprocess; self.device = device
        self.ctm = ctm; self.logit_scale = logit_scale
        self._acts = self._grads = None; self._active = False
        blk = model.visual.transformer.resblocks[-1]
        self._fh = blk.ln_1.register_forward_hook(self._save_acts)
        self._bh = blk.ln_1.register_full_backward_hook(self._save_grads)
    def _save_acts(self, _m, _i, o):
        if self._active: self._acts = o
    def _save_grads(self, _m, _gi, go):
        if self._active: self._grads = go[0]
    def _patch_tokens(self, x):
        if x is None or x.dim() != 3: return None
        if x.shape[1] == 1:   return x[1:, 0, :]
        elif x.shape[0] == 1: return x[0, 1:, :]
        return None
    def _cam(self):
        a = self._patch_tokens(self._acts); g = self._patch_tokens(self._grads)
        if a is None or g is None: return None
        a = a.float(); g = g.float()
        c = F.relu((a * g.mean(0).unsqueeze(0)).sum(-1))
        if c.max() <= 1e-8: return None
        c = c / c.max(); s = int(math.sqrt(c.numel()))
        if s * s != c.numel(): return None
        return F.interpolate(c.reshape(s, s).unsqueeze(0).unsqueeze(0),
                             size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear",
                             align_corners=False).squeeze().detach()
    @torch.enable_grad()
    def _forward_backward(self, pil, backward_fn):
        self._acts = self._grads = None; self._active = True
        t = self.preprocess(pil).unsqueeze(0).to(self.device)
        self.model.zero_grad(set_to_none=True)
        f = self.model.encode_image(t).float(); f = f / f.norm(dim=-1, keepdim=True)
        backward_fn(f).backward(); self._active = False
        return self._cam()
    def compute_combined(self, pil, qv, tidx):
        qv_d = qv.to(self.device).float(); qv_d = qv_d / qv_d.norm()
        sc = self._forward_backward(pil, lambda f: (f * qv_d.unsqueeze(0)).sum(-1).squeeze())
        cc = self._forward_backward(pil, lambda f: (
            self.logit_scale.float() * (f @ self.ctm.float().T)).softmax(-1)[0, tidx])
        vs = sc is not None and sc.max().item() > 1e-6
        vc = cc is not None and cc.max().item() > 1e-6
        if vs and vc:
            r = GRADCAM_SNIPPET_W * (sc / sc.max()) + GRADCAM_CLASS_W * (cc / cc.max())
            return r / r.max() if r.max().item() >= 0.05 else None
        return (sc / sc.max()) if vs else ((cc / cc.max()) if vc else None)
    def remove_hooks(self):
        self._fh.remove(); self._bh.remove()

# ==================== SECTION 4.5 — scoring ==============================
@torch.no_grad()
def zeroshot_probs_gpu(embs, ctm, ls):
    """Legacy 5-way softmax across the target classes. Kept for reference/compat.
    NOTE [v14 FIX A]: this couples co-occurring findings (a patch showing both
    effusion and atelectasis suppresses BOTH) — do NOT use it as the class score
    when USE_BINARY_PROMPTS is on; use binary_class_probs_gpu instead."""
    return (ls.float() * (embs.float() @ ctm.float().T)).softmax(-1)

@torch.no_grad()
def binary_class_probs_gpu(embs, pos_mat, neg_mat, ls):
    """[v14 FIX A] Independent per-class P(finding present) -> [N, C].
    For each class c: softmax( ls * [sim(emb,pos_c), sim(emb,neg_c)] )[...,0].
    Decoupled across classes — the multi-label-correct CheXzero scoring."""
    e = embs.float()
    sp = e @ pos_mat.float().T          # [N, C]
    sn = e @ neg_mat.float().T          # [N, C]
    z = torch.stack([sp, sn], dim=-1) * ls.float()   # [N, C, 2]
    return z.softmax(dim=-1)[..., 0]                  # [N, C] P(present)

@torch.no_grad()
def class_probs_gpu(embs, ctm, ls, pos_mat=None, neg_mat=None):
    """Dispatcher used everywhere downstream. Binary pairs when available."""
    if USE_BINARY_PROMPTS and pos_mat is not None and neg_mat is not None:
        return binary_class_probs_gpu(embs, pos_mat, neg_mat, ls)
    return zeroshot_probs_gpu(embs, ctm, ls)

@torch.no_grad()
def global_image_class_probs(pil, model, preprocess, device, ctm, ls, pos_mat=None, neg_mat=None):
    """[v14 FIX B] Whole-image CheXzero class probabilities -> [C] tensor.
    Patch-max discards global context (esp. Cardiomegaly); the final class score
    ensembles this with the patch score in classify_patches_gpu."""
    t = preprocess(pil).unsqueeze(0).to(device)
    with torch.amp.autocast("cuda", enabled=USE_CUDA):
        f = model.encode_image(t).float(); f = f / f.norm(dim=-1, keepdim=True)
    return class_probs_gpu(f, ctm, ls, pos_mat, neg_mat)[0]   # [C]

print("[cell 4] CheXzero loader, binary prompts, GradCAM, decoupled scoring ready.")


In [ ]:
# =============================================================================
# CELL 5 / 8  —  PATCH UTILITIES (geometry, encode, score, classify, persistence)
# =============================================================================
#   SECTION 5.1  box geometry (_clamp_box_square, extract, dedup, top-k)
#   SECTION 5.2  combined score + patch encoding
#   SECTION 5.3  GradCAM-in-box scoring
#   SECTION 5.4  causal containment test
#   SECTION 5.5  classify_patches_gpu  [v14 FIX A/B: binary prob + global ensemble]
#   SECTION 5.6  spurious-in-anatomy selection
#   SECTION 5.7  persistent-homology causal mask
#   SECTION 5.8  outside-anatomy scan
# -----------------------------------------------------------------------------

# ==================== SECTION 5.1 — box geometry ==========================
def _clamp_box_square(x1, y1, side, W=IMAGE_SIZE, H=IMAGE_SIZE, min_side=MIN_BOX_SIZE):
    side = int(round(side))
    if side < min_side: side = min_side
    if side > min(W, H): side = min(W, H)
    x1 = int(max(0, min(int(round(x1)), W - side)))
    y1 = int(max(0, min(int(round(y1)), H - side)))
    x2, y2 = x1 + side, y1 + side
    if x2 <= x1 or y2 <= y1: return None
    return (x1, y1, x2, y2)

def extract_candidate_boxes_gpu(hmap, scale, stride, thresh):
    p = F.avg_pool2d(hmap.unsqueeze(0).unsqueeze(0).float(),
                     kernel_size=scale, stride=stride, padding=0).squeeze()
    v = (p >= thresh).nonzero(as_tuple=False)
    if v.shape[0] == 0: return []
    y1s = (v[:, 0] * stride).cpu().tolist(); x1s = (v[:, 1] * stride).cpu().tolist()
    out = []
    for x, y in zip(x1s, y1s):
        b = _clamp_box_square(x, y, scale)
        if b is not None: out.append(b)
    return out

def deduplicate_boxes(boxes):
    seen, u = set(), []
    for b in boxes:
        k = tuple(map(int, b))
        if k not in seen: seen.add(k); u.append(b)
    return u

def keep_top_boxes_gpu(boxes, hmap, mx):
    if len(boxes) <= mx: return boxes
    sc = boxes[0][2] - boxes[0][0]
    p = F.avg_pool2d(hmap.unsqueeze(0).unsqueeze(0).float(),
                     kernel_size=sc, stride=1, padding=0).squeeze()
    pH, pW = p.shape
    scores = [float(p[min(b[1], pH - 1), min(b[0], pW - 1)].item()) for b in boxes]
    idx = torch.tensor(scores).topk(mx).indices.tolist()
    return [boxes[i] for i in sorted(idx)]

# ==================== SECTION 5.2 — score + encode ========================
def combined_score_gpu(sem, prob, gc):
    eps = 1e-6
    s01 = ((sem + 1.0) * 0.5).clamp(eps, 1.0)
    return (s01 ** SCORE_W_SEM * prob.clamp(eps, 1.0) ** SCORE_W_PROB
            * gc.clamp(eps, 1.0) ** SCORE_W_GC).float()

@torch.no_grad()
def encode_patch_batch_gpu(crops, model, preprocess, device, bs=PATCH_ENCODE_BATCH_SZ):
    if not crops: return torch.zeros(0, CLIP_EMBED_DIM, device=device)
    outs = []
    for i in range(0, len(crops), bs):
        batch = torch.stack([preprocess(c) for c in crops[i:i + bs]]).to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_CUDA):
            f = model.encode_image(batch).float(); f = f / f.norm(dim=-1, keepdim=True)
        outs.append(f)
    return torch.cat(outs)

# ==================== SECTION 5.3 — GradCAM in box ========================
def _gc_scores_batch(boxes, gcam_gpu, device):
    if gcam_gpu is None:
        return torch.full((len(boxes),), 0.0, device=device)
    H, W = gcam_gpu.shape[-2], gcam_gpu.shape[-1]
    scores = torch.empty(len(boxes), device=device)
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        xa, ya = max(0, int(x1)), max(0, int(y1))
        xb, yb = min(W, int(x2)), min(H, int(y2))
        if xb <= xa or yb <= ya:
            scores[i] = float(GRADCAM_WEAK_THR); _COUNTERS["gc_degenerate_box"] += 1; continue
        scores[i] = gcam_gpu[ya:yb, xa:xb].mean()
    return torch.nan_to_num(scores, nan=float(GRADCAM_WEAK_THR))

# ==================== SECTION 5.4 — causal containment ====================
def _causal_contains(box, cx_p, cy_p, mask, peaks):
    x1, y1, x2, y2 = box
    center_in = bool(mask[min(cy_p, IMAGE_SIZE - 1), min(cx_p, IMAGE_SIZE - 1)].item() > 0.5)
    if CAUSAL_CONTAIN_MODE == "center":
        return center_in
    peak_in = any(x1 <= px < x2 and y1 <= py < y2 for (py, px) in peaks) if peaks else False
    if CAUSAL_CONTAIN_MODE == "peak":
        return center_in or peak_in
    sub = mask[y1:y2, x1:x2]
    cover = (float(sub.sum().item()) / max(float(mask.sum().item()), 1.0)) if sub.numel() else 0.0
    if CAUSAL_CONTAIN_MODE == "cover":
        return center_in or cover >= CAUSAL_MASK_COVER_FRAC
    return center_in or peak_in or cover >= CAUSAL_MASK_COVER_FRAC

# ==================== SECTION 5.5 — classify patches ======================
def classify_patches_gpu(boxes, embs_gpu, probs_gpu, gcam_gpu, qi, target_idx,
                         zoom_level, image_name, device,
                         persistent_mask=None, persistent_peaks=None, global_prob=None):
    """Label each in-anatomy candidate causal / not.
    probs_gpu: [N, C] independent class probs (binary pairs) from class_probs_gpu.
    global_prob: [C] whole-image probs; [v14 FIX B] the stored/used class score is
      ens = GLOBAL_ENSEMBLE_ALPHA*global + (1-alpha)*patch  for the target class."""
    if not boxes or qi.query_vector is None:
        return [], []
    qv = qi.query_vector.to(device).float(); qv = qv / qv.norm()
    sims = (embs_gpu.float() * qv.unsqueeze(0)).sum(-1)
    patch_prob = probs_gpu[:, target_idx]
    # [v14 FIX B] ensemble the patch class prob with the whole-image class prob
    if global_prob is not None and GLOBAL_ENSEMBLE_ALPHA > 0.0:
        g = float(global_prob[target_idx].item())
        cls_prob = GLOBAL_ENSEMBLE_ALPHA * g + (1.0 - GLOBAL_ENSEMBLE_ALPHA) * patch_prob
    else:
        cls_prob = patch_prob
    gc_vals = _gc_scores_batch(boxes, gcam_gpu, device)
    combo = combined_score_gpu(sims, cls_prob, gc_vals)
    sem_thr = SEMANTIC_THRESHOLD_PER_CLASS.get(qi.pathology, SEMANTIC_THRESHOLD)
    region = qi.anatomical_prior.region_name if qi.anatomical_prior else "unknown"
    peaks = persistent_peaks or []
    sims_cpu = sims.cpu().numpy(); probs_cpu = cls_prob.cpu().numpy()
    gc_cpu = gc_vals.cpu().numpy(); combo_cpu = combo.cpu().numpy()
    causal, all_scored = [], []
    for i, box in enumerate(boxes):
        sim = float(sims_cpu[i]); prob = float(probs_cpu[i])
        gc_v = float(gc_cpu[i]); cv = float(combo_cpu[i])
        cx_p = (box[0] + box[2]) // 2; cy_p = (box[1] + box[3]) // 2
        has_sem = sim >= sem_thr
        if persistent_mask is not None:
            is_causal = has_sem and _causal_contains(box, cx_p, cy_p, persistent_mask, peaks)
        else:
            is_causal = has_sem and gc_v >= GRADCAM_WEAK_THR
        doc = PatchDocument(
            image_name=image_name, pathology=qi.pathology, scale=int(box[2] - box[0]),
            box=tuple(map(int, box)), visual_embedding=embs_gpu[i].detach().cpu().float(),
            semantic_score=sim, zeroshot_prob=prob, gradcam_score=gc_v, combined_score=cv,
            causal=is_causal, anatomical_region=region, confidence=float(qi.confidence),
            text_snippet=str(qi.text_snippet), zoom_level=zoom_level,
            spurious_source="in_anatomy", selection_source="threshold")
        all_scored.append(doc)
        if is_causal: causal.append(doc)
    return causal, all_scored

def select_spurious_in_anatomy(all_scored, sem_thr, max_keep=SPUR_IN_MAX_KEEP):
    noncausal = [d for d in all_scored if not d.causal]
    if not noncausal: return []
    seen, nc = set(), []
    for d in sorted(noncausal, key=lambda x: x.semantic_score):
        if d.box not in seen: seen.add(d.box); nc.append(d)
    gated = [d for d in nc if d.semantic_score < sem_thr and d.gradcam_score >= SPUR_IN_GC_FLOOR]
    if not gated:
        gated = nc[:max_keep]; _COUNTERS["spur_in_safetynet"] += 1
    spin = gated[:max_keep]
    for d in spin:
        d.causal = False; d.spurious_source = "in_anatomy"; d.selection_source = "threshold"
    return spin

# ==================== SECTION 5.6/5.7 — persistence mask ==================
def _persistent_mask_scipy(gcam_np, top_k=PERSISTENCE_TOP_K, n_levels=PERSISTENCE_N_LEVELS):
    H, W = gcam_np.shape
    vmax = float(gcam_np.max()); vmin = float(max(gcam_np.min(), 0.0))
    if vmax - vmin < 0.01: return None, []
    thresholds = np.linspace(vmax * 0.98, vmin + 1e-4, n_levels)
    alive = {}; dead = []; nxt = 0
    prev_labels = np.zeros((H, W), dtype=np.int32); prev_map = {}
    for t in thresholds:
        binary = (gcam_np >= t); labels, n = ndimage_label(binary); curr_map = {}
        for c in range(1, n + 1):
            cmask = (labels == c); overlap = prev_labels[cmask]; parents = set()
            for pl in np.unique(overlap):
                if pl > 0 and pl in prev_map: parents.add(prev_map[pl])
            if len(parents) == 0:
                vals = gcam_np.copy(); vals[~cmask] = -1
                peak = np.unravel_index(vals.argmax(), (H, W))
                alive[nxt] = {"birth": float(t), "peak": peak}; curr_map[c] = nxt; nxt += 1
            elif len(parents) == 1:
                curr_map[c] = list(parents)[0]
            else:
                sp = sorted(parents, key=lambda i: alive[i]["birth"], reverse=True); elder = sp[0]
                for younger in sp[1:]:
                    if younger in alive:
                        dead.append((alive[younger]["birth"] - t, alive[younger]["birth"],
                                     alive[younger]["peak"])); del alive[younger]
                curr_map[c] = elder
        prev_labels = labels; prev_map = curr_map
    for cid, info in alive.items():
        dead.append((info["birth"] - vmin, info["birth"], info["peak"]))
    if not dead: return None, []
    dead.sort(key=lambda x: -x[0])
    mask = np.zeros((H, W), dtype=bool); peaks = []
    for k in range(min(top_k, len(dead))):
        pers, birth, peak_yx = dead[k]
        if pers < 0.01: continue
        candidate = (gcam_np >= birth * 0.95); labels, n = ndimage_label(candidate)
        if n == 0: continue
        py, px = peak_yx; cid = labels[py, px]
        if cid > 0:
            comp = (labels == cid)
            if comp.sum() >= PERSISTENCE_MIN_AREA: mask |= comp; peaks.append((int(py), int(px)))
    if mask.sum() < PERSISTENCE_MIN_AREA: return None, []
    return mask, peaks

def compute_persistent_causal_mask(gcam_gpu):
    if gcam_gpu is None or not PERSISTENCE_ENABLED: return None, []
    gcam_np = gcam_gpu.cpu().numpy().astype(np.float64)
    if gcam_np.max() - gcam_np.min() < 0.01: return None, []
    mask, peaks = _persistent_mask_scipy(gcam_np)
    if mask is None: return None, []
    if CAUSAL_MASK_DILATE_PX > 0:
        mask = binary_dilation(mask, iterations=int(CAUSAL_MASK_DILATE_PX))
    _COUNTERS["persistence_used"] += 1
    return torch.from_numpy(mask.astype(np.float32)).to(gcam_gpu.device), peaks

# ==================== SECTION 5.8 — outside-anatomy scan ==================
def scan_all_outside_anatomy(pil_img, plan, model, preprocess, device, ctm_gpu, ls_gpu, composite_hmap):
    inv_hmap = (1.0 - composite_hmap.clamp(0, 1)).clamp(0, 1)
    qvs = [qi.query_vector for qi in plan.query_items
           if qi.query_vector is not None and not qi.negated]
    qmat = (torch.stack([q.to(device).float() / q.to(device).float().norm() for q in qvs])
            if qvs else None)
    gray = np.asarray(pil_img.convert("L"), dtype=np.float32) / 255.0
    gH, gW = gray.shape[0], gray.shape[1]
    def _score_boxes(boxes):
        if not boxes: return []
        crops = [pil_img.crop(b) for b in boxes]
        embs = encode_patch_batch_gpu(crops, model, preprocess, device)
        if qmat is not None:
            sims = (embs.float() @ qmat.T).max(dim=1).values
        else:
            sims = torch.zeros(embs.shape[0], device=device)
        sims_cpu = sims.cpu().numpy(); out = []
        for i, b in enumerate(boxes):
            x1, y1, x2, y2 = b
            xa, ya = max(0, int(x1)), max(0, int(y1))
            xb, yb = min(gW, int(x2)), min(gH, int(y2))
            patch = gray[ya:yb, xa:xb]
            if patch.size == 0:
                _COUNTERS["outside_degenerate_box"] += 1; continue
            out.append({"box": b, "emb": embs[i].detach().cpu().float(),
                        "sim": float(sims_cpu[i]), "mean": float(patch.mean()),
                        "std": float(patch.std())})
        return out
    geo_boxes = []
    for sc in OUTSIDE_ANAT_SCALES:
        stride = max(16, STRIDE_BASE * sc // 128)
        geo_boxes += extract_candidate_boxes_gpu(inv_hmap, sc, stride, OUTSIDE_ANAT_INV_THRESH)
    geo_boxes = deduplicate_boxes(geo_boxes)[:OUTSIDE_ANAT_MAX_CAND]
    border = torch.zeros(IMAGE_SIZE, IMAGE_SIZE, device=device)
    bw = int(IMAGE_SIZE * ARTIFACT_BORDER_FRAC)
    border[:bw, :] = 1.0; border[-bw:, :] = 1.0; border[:, :bw] = 1.0; border[:, -bw:] = 1.0
    ap = (border * inv_hmap).clamp(0, 1)
    int_boxes = []
    for sc in ARTIFACT_SCALES:
        stride = max(16, STRIDE_BASE * sc // 128)
        int_boxes += extract_candidate_boxes_gpu(ap, sc, stride, 0.35)
    int_boxes = deduplicate_boxes(int_boxes)[:OUTSIDE_ANAT_MAX_CAND]
    geo_scored = _score_boxes(geo_boxes); int_scored = _score_boxes(int_boxes)
    def _select(scored, sim_cap, max_keep, sel_src):
        if not scored: return []
        gated = [s for s in scored if s["sim"] < sim_cap]
        if not gated:
            gated = sorted(scored, key=lambda s: s["sim"])[:max_keep]
            _COUNTERS[f"outside_{sel_src}_safetynet"] += 1
        gated = sorted(gated, key=lambda s: s["std"], reverse=True)[:max_keep]
        docs = []
        for s in gated:
            b = s["box"]
            docs.append(PatchDocument(
                image_name=plan.image_name, pathology="domain_artifact",
                scale=int(b[2] - b[0]), box=tuple(map(int, b)), visual_embedding=s["emb"],
                semantic_score=s["sim"], zeroshot_prob=0.0, gradcam_score=s["std"],
                combined_score=s["mean"], causal=False, anatomical_region="outside_anatomy",
                confidence=1.0, text_snippet=f"{sel_src}_m={s['mean']:.3f}_s={s['std']:.3f}",
                zoom_level=1, spurious_source="outside_anatomy", selection_source=sel_src))
        return docs
    geo_docs = _select(geo_scored, OUTSIDE_ANAT_SIM_CAP, OUTSIDE_ANAT_MAX_KEEP, "geometric_scan")
    int_docs = _select(int_scored, ARTIFACT_SIM_CAP, ARTIFACT_MAX_KEEP, "intensity_scan")
    seen = set(d.box for d in geo_docs); merged = list(geo_docs)
    for d in int_docs:
        if d.box not in seen: seen.add(d.box); merged.append(d)
    if merged:
        _COUNTERS["outside_anatomy_patches"] += len(merged)
        _COUNTERS["images_with_outside_anatomy"] += 1
    return merged[:OUTSIDE_ANAT_MAX_KEEP + ARTIFACT_MAX_KEEP]

print("[cell 5] patch geometry, binary+ensemble classifier, persistence, outside-scan ready.")


In [ ]:
# =============================================================================
# CELL 6 / 8  —  STAGE-3 DISCOVERY  +  A* METRICS  +  GRADCAM / BBOX VISUALS
# =============================================================================
#   SECTION 6.1  discover_patches_for_plan  [threads binary probs + global ensemble]
#   SECTION 6.2  run_stage3 resumable driver
#   SECTION 6.3  split embeddings 3 ways + summarize
#   SECTION 6.4  A* localization metrics (pointing-game, IoU@0.1/0.25) from BBoxes
#   SECTION 6.5  A* image-level metrics (AUROC + AUPRC vs prevalence)
#   SECTION 6.6  GradCAM grid for CAUSAL patches — 5 classes  (must-have figure)
#   SECTION 6.7  bounding boxes shown SEPARATELY per bucket (causal/spur-in/spur-out)
# -----------------------------------------------------------------------------

# ==================== SECTION 6.1 — discover ==============================
def discover_patches_for_plan(plan, gradcam, model, preprocess, device, ctm, ls,
                              split_name="", pos_mat=None, neg_mat=None):
    result = ImagePatchResult(image_name=plan.image_name, image_path=plan.image_path, split=split_name)
    if not plan.image_path or not os.path.isfile(plan.image_path):
        _COUNTERS["images_missing"] += 1; return result
    if not plan.query_items:
        return result
    try:
        pil = Image.open(plan.image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
    except Exception:
        _COUNTERS["images_failed"] += 1; return result

    cmap = {c.lower(): c for c in TARGET_CLASSES}
    comp_hmap = get_composite_heatmap_gpu(plan, IMAGE_SIZE, IMAGE_SIZE, device)
    # [v14 FIX B] one whole-image class-prob vector, ensembled into every patch
    global_prob = global_image_class_probs(pil, model, preprocess, device, ctm, ls, pos_mat, neg_mat)
    rc, rs = [], []

    for qi in plan.query_items:
        if qi.negated or qi.query_vector is None: continue
        key = qi.pathology.lower().strip()
        if key not in cmap: continue
        qi.pathology = cmap[key]; tidx = TARGET_CLASSES.index(qi.pathology)
        sem_thr = SEMANTIC_THRESHOLD_PER_CLASS.get(qi.pathology, SEMANTIC_THRESHOLD)

        if qi.anatomical_prior:
            p = qi.anatomical_prior
            ihmap = make_gaussian_heatmap_gpu(p.center_x, p.center_y, p.sigma_x, p.sigma_y,
                                              IMAGE_SIZE, IMAGE_SIZE, device)
        else:
            ihmap = comp_hmap.clone()
        mx = ihmap.max()
        if mx > 0: ihmap = ihmap / mx

        gcam = gradcam.compute_combined(pil, qi.query_vector, tidx) if ENABLE_GRADCAM else None
        pmask, ppeaks = compute_persistent_causal_mask(gcam)
        if pmask is None: _COUNTERS["persistence_fallback"] += 1

        ac, asc = [], []
        scales = _SCALE_MAP.get(qi.pathology, PATCH_SCALES)   # [v14 FIX C] per-class scales
        for it in range(1, MAX_ITER + 1):
            sp = SPATIAL_THRESH * (0.70 ** (it - 1)); ih = ihmap.clone()
            if it > 1:
                ih = (ih * 1.30).clamp(0, 1); result.refined = True
            for sc in scales:
                stride = max(16, STRIDE_BASE * sc // 128)
                boxes = keep_top_boxes_gpu(
                    deduplicate_boxes(extract_candidate_boxes_gpu(ih, sc, stride, sp)),
                    ih, MAX_CANDIDATE_BOXES)
                if not boxes: continue
                embs = encode_patch_batch_gpu([pil.crop(b) for b in boxes], model, preprocess, device)
                probs = class_probs_gpu(embs, ctm, ls, pos_mat, neg_mat)  # [v14 FIX A]
                c, a = classify_patches_gpu(boxes, embs, probs, gcam, qi, tidx, it,
                                            plan.image_name, device,
                                            persistent_mask=pmask, persistent_peaks=ppeaks,
                                            global_prob=global_prob)
                ac.extend(c); asc.extend(a)
            if len(ac) >= MIN_CAUSAL_PATCHES:
                if max((d.zeroshot_prob for d in ac), default=0) >= CONF_THRESHOLD or it == MAX_ITER:
                    break

        if 0 < len(ac) < MIN_CAUSAL_PATCHES and ENABLE_ZOOM_REFINEMENT:
            top = sorted(ac, key=lambda d: d.combined_score, reverse=True)[0]
            x1, y1, x2, y2 = top.box
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2; w, h = x2 - x1, y2 - y1
            ex1 = max(0, int(cx - w * 0.75)); ey1 = max(0, int(cy - h * 0.75))
            ex2 = min(IMAGE_SIZE, int(cx + w * 0.75)); ey2 = min(IMAGE_SIZE, int(cy + h * 0.75))
            if ex2 > ex1 and ey2 > ey1:
                sxr = (ex2 - ex1) / IMAGE_SIZE; syr = (ey2 - ey1) / IMAGE_SIZE
                iso = 0.5 * (sxr + syr)
                zhmap = torch.ones(IMAGE_SIZE, IMAGE_SIZE, device=device); remap_boxes = []
                for sc in ZOOM_SCALES:
                    stride = max(12, STRIDE_BASE * sc // 128)
                    zb = deduplicate_boxes(extract_candidate_boxes_gpu(zhmap, sc, stride, 0.0))[:MAX_ZOOM_CANDIDATE_BOXES]
                    for b in zb:
                        zcx = 0.5 * (b[0] + b[2]); zcy = 0.5 * (b[1] + b[3])
                        ocx = ex1 + zcx * sxr; ocy = ey1 + zcy * syr
                        ob = _clamp_box_square(ocx - sc * iso / 2, ocy - sc * iso / 2, sc * iso)
                        if ob is not None: remap_boxes.append(ob)
                remap_boxes = deduplicate_boxes(remap_boxes)
                if remap_boxes:
                    ze = encode_patch_batch_gpu([pil.crop(b) for b in remap_boxes], model, preprocess, device)
                    zp = class_probs_gpu(ze, ctm, ls, pos_mat, neg_mat)
                    zc, za = classify_patches_gpu(remap_boxes, ze, zp, gcam, qi, tidx, 2,
                                                  plan.image_name, device,
                                                  persistent_mask=pmask, persistent_peaks=ppeaks,
                                                  global_prob=global_prob)
                    ac.extend(zc); asc.extend(za)
                result.refined = True

        if ALLOW_FALLBACK and not ac and asc:
            pool = [d for d in asc if d.semantic_score >= sem_thr * 0.6]
            if pool:
                fb = sorted(pool, key=lambda d: d.combined_score, reverse=True)[:FALLBACK_TOP_K]
                for d in fb: d.causal = True; d.selection_source = "fallback"
                ac.extend(fb); result.used_fallback = True

        spin = select_spurious_in_anatomy(asc, sem_thr, max_keep=SPUR_IN_MAX_KEEP)
        rs.extend(spin)
        rc.extend(sorted(ac, key=lambda d: d.combined_score, reverse=True)[:TOP_K_PER_FIND])

    rs.extend(scan_all_outside_anatomy(pil, plan, model, preprocess, device, ctm, ls, comp_hmap))
    result.causal_patches = rc; result.spurious_patches = rs
    result.n_iterations = MAX_ITER if result.refined else 1
    return result

# ==================== SECTION 6.2 — run_stage3 driver =====================
def run_stage3(plans, split_name, gradcam, model, preprocess, device, ctm, ls,
               deadline=None, out_dir=".", pos_mat=None, neg_mat=None):
    ckpt = f"{out_dir}/stage3_{split_name}_ckpt.pkl"; start = 0; results = []
    if os.path.exists(ckpt):
        with open(ckpt, "rb") as f: results = pickle.load(f)
        start = len(results)
        print(f"  Resuming {split_name} from checkpoint: {start:,}/{len(plans):,}")
    n_fail = 0; n_empty = sum(1 for r in results if not r.causal_patches)
    t0 = time.time(); last_ckpt_t = t0; stopped_early = False
    pbar = tqdm(range(start, len(plans)), desc=f"Stage 3 - {split_name}", initial=start, total=len(plans))
    for idx in pbar:
        if deadline is not None and time.time() > deadline:
            _atomic_pickle(results, ckpt); stopped_early = True
            print(f"\n  [{split_name}] Budget reached at {idx:,}/{len(plans):,}. Checkpoint saved."); break
        try:
            res = discover_patches_for_plan(plans[idx], gradcam, model, preprocess, device, ctm, ls,
                                            split_name, pos_mat=pos_mat, neg_mat=neg_mat)
        except Exception as e:
            print(f"  {plans[idx].image_name}: {e}")
            res = ImagePatchResult(image_name=plans[idx].image_name,
                                   image_path=plans[idx].image_path, split=split_name)
            n_fail += 1
        results.append(res)
        if not res.causal_patches: n_empty += 1
        now = time.time(); done_now = len(results) - start
        if (done_now % CKPT_EVERY_IMAGES == 0) or ((now - last_ckpt_t) > CKPT_EVERY_MIN * 60):
            if USE_CUDA: torch.cuda.empty_cache()
            gc.collect(); _atomic_pickle(results, ckpt); last_ckpt_t = now
            rate = done_now / max(now - t0, 1e-6); remaining = len(plans) - len(results)
            eta_h = remaining / max(rate, 1e-9) / 3600.0
            pbar.set_postfix_str(f"{rate*3600:,.0f} img/h | ETA {eta_h:.2f}h | budget {_budget_left_h(deadline):.2f}h")
    _atomic_pickle(results, ckpt); completed = (len(results) >= len(plans))
    if completed:
        print(f"  {split_name}: {n_fail} failed | {n_empty:,} empty | {len(results)-n_empty:,} w/causal | complete")
        if os.path.exists(ckpt): os.remove(ckpt)
    else:
        print(f"  {split_name}: partial {len(results):,}/{len(plans):,} ({len(results)-n_empty:,} w/causal). Checkpoint kept.")
    return results, completed, stopped_early

# ==================== SECTION 6.3 — split + summarize =====================
_MK = ["image_name", "pathology", "box", "scale", "semantic_score", "zeroshot_prob",
       "gradcam_score", "combined_score", "anatomical_region", "confidence",
       "text_snippet", "zoom_level"]
def _md(p, sp, ek):
    d = {"split": sp}
    for k in _MK: d[k] = getattr(p, k)
    for k in ek: d[k] = getattr(p, k)
    return d

def split_embeddings_three_ways(rd):
    ce, cm = [], []; se, sm = [], []; oe, om = [], []
    for sp, results in rd.items():
        for r in results:
            for p in r.causal_patches:
                ce.append(p.visual_embedding.float()); cm.append(_md(p, sp, ["selection_source"]))
            for p in r.spurious_patches:
                if p.spurious_source == "outside_anatomy":
                    oe.append(p.visual_embedding.float()); om.append(_md(p, sp, ["spurious_source", "selection_source"]))
                else:
                    se.append(p.visual_embedding.float()); sm.append(_md(p, sp, ["spurious_source", "selection_source"]))
    def pk(e, m):
        return {"embeddings": torch.stack(e) if e else torch.zeros(0, CLIP_EMBED_DIM), "meta": m, "n": len(e)}
    return pk(ce, cm), pk(se, sm), pk(oe, om)

def summarize_stage3(results, split):
    nc = sum(len(r.causal_patches) for r in results)
    ne = sum(1 for r in results if r.causal_patches)
    nfb = sum(sum(1 for p in r.causal_patches if p.selection_source == "fallback") for r in results)
    nfi = sum(1 for r in results if r.used_fallback); n_true = nc - nfb
    nsi = sum(sum(1 for p in r.spurious_patches if p.spurious_source == "in_anatomy") for r in results)
    nso = sum(sum(1 for p in r.spurious_patches if p.spurious_source == "outside_anatomy") for r in results)
    fb_pct = 100.0 * nfb / max(nc, 1)
    print(f"{split:<6}: {len(results):,} imgs | {ne:,} w/causal | {nc:,} causal "
          f"({n_true} true / {nfb} fb = {fb_pct:.1f}% fb / {nfi} imgs) | spur_in={nsi:,} spur_out={nso:,}")

# ==================== SECTION 6.4 — A* localization metrics ===============
# NIH label -> BBox_List_2017 "Finding Label"
_BBOX_LABEL_MAP = {"Atelectasis": "Atelectasis", "Cardiomegaly": "Cardiomegaly",
                   "Pleural Effusion": "Effusion"}   # Consolidation/Edema: no GT boxes

def load_bbox_gt(bbox_csv, orig_size=1024):
    """Return {basename: [(class, (x1,y1,x2,y2) in IMAGE_SIZE space), ...]}."""
    if not bbox_csv or not os.path.exists(bbox_csv):
        print(f"  [loc] BBox CSV not found ({bbox_csv}) — localization eval skipped."); return {}
    df = pd.read_csv(bbox_csv)
    cols = {c.lower().strip(): c for c in df.columns}
    ci = cols.get("image index", "Image Index"); cl = cols.get("finding label", "Finding Label")
    bx = next((cols[c] for c in cols if c.startswith("bbox [x")), None) or "Bbox [x"
    by = [c for c in df.columns if c.strip() in ("y", "y]")]
    bw = [c for c in df.columns if c.strip() in ("w", "w]")]
    bh = [c for c in df.columns if c.strip() in ("h", "h]")]
    by = by[0] if by else "y"; bw = bw[0] if bw else "w"; bh = bh[0] if bh else "h]"
    inv = {v: k for k, v in _BBOX_LABEL_MAP.items()}
    s = IMAGE_SIZE / float(orig_size); gt = defaultdict(list)
    for _, r in df.iterrows():
        lab = str(r[cl]).strip()
        if lab not in inv: continue
        try:
            x = float(r[bx]) * s; y = float(r[by]) * s; w = float(r[bw]) * s; h = float(r[bh]) * s
        except Exception:
            continue
        gt[os.path.basename(str(r[ci]))].append((inv[lab], (x, y, x + w, y + h)))
    print(f"  [loc] loaded {sum(len(v) for v in gt.values())} GT boxes over {len(gt)} images "
          f"(classes: {sorted(set(inv.values()))})")
    return gt

def _iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1); ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1); inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0

def _center_in(box, gt):
    cx, cy = (box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0
    return gt[0] <= cx <= gt[2] and gt[1] <= cy <= gt[3]

def evaluate_localization(rd, bbox_gt, iou_thrs=(0.1, 0.25), causal_only=True):
    """A* patch-quality metric: does the top causal patch land on the GT lesion?
    Pointing-game accuracy + IoU@thr, per class, over images with GT + a patch."""
    if not bbox_gt:
        return pd.DataFrame()
    results = rd.get("train", []) + rd.get("val", [])
    per = {c: {"n": 0, "hit": 0, **{f"iou@{t}": 0 for t in iou_thrs}} for c in _BBOX_LABEL_MAP}
    for r in results:
        gts = bbox_gt.get(os.path.basename(str(r.image_name)))
        if not gts: continue
        for cls in _BBOX_LABEL_MAP:
            cls_gts = [g[1] for g in gts if g[0] == cls]
            if not cls_gts: continue
            cand = [p for p in r.causal_patches if p.pathology == cls and
                    (not causal_only or p.selection_source != "fallback")]
            if not cand:
                cand = [p for p in r.causal_patches if p.pathology == cls]  # allow fallback if nothing else
            if not cand: continue
            best = max(cand, key=lambda p: p.combined_score); pb = best.box
            per[cls]["n"] += 1
            if any(_center_in(pb, g) for g in cls_gts): per[cls]["hit"] += 1
            miou = max(_iou(pb, g) for g in cls_gts)
            for t in iou_thrs:
                if miou >= t: per[cls][f"iou@{t}"] += 1
    rows = []
    for c, d in per.items():
        n = max(d["n"], 1)
        row = {"class": c, "n_eval": d["n"], "pointing_game": d["hit"] / n}
        for t in iou_thrs: row[f"iou@{t}"] = d[f"iou@{t}"] / n
        rows.append(row)
    df = pd.DataFrame(rows)
    if not df.empty:
        print("\n" + "=" * 66 + "\nA* LOCALIZATION (top causal patch vs NIH GT box)\n" + "=" * 66)
        print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
        print("pointing_game = center-in-GT rate; iou@t = fraction with IoU>=t. Higher is better.")
    return df

# ==================== SECTION 6.5 — A* image-level metrics ================
def build_gt_lookup(train_df, val_df):
    lut = {}
    for df in (train_df, val_df):
        if df is None: continue
        lcols = _find_label_cols(df)
        name_col = next((c for c in ["image_name", "Image Index", "image"] if c in df.columns), None)
        for _, row in df.iterrows():
            nm = os.path.basename(str(row[name_col])) if name_col else os.path.basename(_resolve_image_name(row))
            vec = {}
            for c in TARGET_CLASSES:
                col = lcols.get(c)
                v = row[col] if col is not None else _get_label_val(row, c)
                try:
                    fv = float(v) if (v is not None and not pd.isna(v)) else float("nan")
                    vec[c] = fv if fv in (0.0, 1.0) else float("nan")
                except Exception:
                    vec[c] = float("nan")
            lut[nm] = vec
    return lut

def image_level_metrics(rd, gt_lut, topk=3):
    """Per-class AUROC + AUPRC (vs prevalence). Score = max/top-k mean of the
    ensembled causal-patch class prob per image. AUPRC<prevalence => below chance."""
    results = rd.get("train", []) + rd.get("val", [])
    scores = {c: [] for c in TARGET_CLASSES}; gts = {c: [] for c in TARGET_CLASSES}
    for r in results:
        gt = gt_lut.get(os.path.basename(str(r.image_name)))
        if gt is None: continue
        by_c = {c: [] for c in TARGET_CLASSES}
        for p in r.causal_patches:
            if p.pathology in by_c: by_c[p.pathology].append(float(p.zeroshot_prob))
        for c in TARGET_CLASSES:
            lv = gt.get(c, float("nan"))
            if lv == lv:
                pr = sorted(by_c[c], reverse=True)
                scores[c].append(float(np.mean(pr[:topk])) if pr else 0.0); gts[c].append(lv)
    rows = []
    for c in TARGET_CLASSES:
        yt = np.asarray(gts[c]); ys = np.asarray(scores[c])
        if len(np.unique(yt)) != 2:
            rows.append({"class": c, "auroc": float("nan"), "auprc": float("nan"),
                         "prevalence": float(yt.mean()) if len(yt) else float("nan"),
                         "n_pos": int((yt == 1).sum()), "n_neg": int((yt == 0).sum())}); continue
        rows.append({"class": c, "auroc": float(roc_auc_score(yt, ys)),
                     "auprc": float(average_precision_score(yt, ys)),
                     "prevalence": float(yt.mean()),
                     "n_pos": int((yt == 1).sum()), "n_neg": int((yt == 0).sum())})
    df = pd.DataFrame(rows)
    if not df.empty:
        df["auprc_gain"] = df["auprc"] - df["prevalence"]   # >0 means better than guessing
        print("\n" + "=" * 78 + "\nA* IMAGE-LEVEL METRICS (ensembled causal-patch score)\n" + "=" * 78)
        print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
        print(f"macro AUROC={np.nanmean(df['auroc']):.3f} | macro AUPRC={np.nanmean(df['auprc']):.3f} "
              f"| auprc_gain<0 => below prevalence (worse than guessing)")
    return df

# ==================== SECTION 6.6 — GradCAM grid for causal patches =======
def plot_causal_gradcam_grid(results, plans_by_name, gradcam, save_path=None):
    """Must-have figure: for each of the 5 classes, one row = (a) original,
    (b) GradCAM heatmap, (c) overlay — computed for the CAUSAL patch's class."""
    import matplotlib.cm as cm
    fig, axes = plt.subplots(N_CLASSES, 3, figsize=(9, 3 * N_CLASSES))
    if N_CLASSES == 1: axes = np.array([axes])
    col_titles = ["(a) chest X-ray", "(b) GradCAM (causal class)", "(c) overlay + causal box"]
    for ridx, cls in enumerate(TARGET_CLASSES):
        pick = None
        for r in results:
            cps = [p for p in r.causal_patches if p.pathology == cls and p.selection_source != "fallback"]
            if not cps: cps = [p for p in r.causal_patches if p.pathology == cls]
            if cps and r.image_path and os.path.isfile(r.image_path):
                pick = (r, max(cps, key=lambda p: p.combined_score)); break
        for c in range(3): axes[ridx, c].axis("off")
        if pick is None:
            axes[ridx, 0].set_ylabel(cls); axes[ridx, 0].set_title(f"{cls}: no causal patch"); continue
        r, patch = pick
        plan = plans_by_name.get(r.image_name)
        qi = None
        if plan is not None:
            qi = next((q for q in plan.query_items if q.pathology == cls and q.query_vector is not None), None)
        pil = Image.open(r.image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
        img = np.asarray(pil).astype(np.float32) / 255.0
        gcam = None
        if qi is not None and gradcam is not None:
            gcam = gradcam.compute_combined(pil, qi.query_vector, TARGET_CLASSES.index(cls))
        heat = gcam.cpu().numpy() if gcam is not None else np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.float32)
        heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
        heat_rgb = cm.jet(heat)[..., :3]
        overlay = 0.55 * img + 0.45 * heat_rgb
        axes[ridx, 0].imshow(img)
        axes[ridx, 1].imshow(heat_rgb)
        axes[ridx, 2].imshow(np.clip(overlay, 0, 1))
        x1, y1, x2, y2 = patch.box
        axes[ridx, 2].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                fill=False, edgecolor="lime", lw=2.2))
        axes[ridx, 0].set_title(cls, fontsize=11, loc="left")
        if ridx == 0:
            for c in range(3): axes[ridx, c].set_title(col_titles[c], fontsize=10)
            axes[ridx, 0].set_title(f"{cls}\n{col_titles[0]}", fontsize=10)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=140, bbox_inches="tight"); print(f"Saved: {save_path}")
    plt.show()

# ==================== SECTION 6.7 — bboxes separated by bucket ============
def plot_bboxes_by_bucket(results, n_examples=4, save_path=None):
    """Three columns: CAUSAL (green/blue) | SPURIOUS-in-anatomy (red) |
    SPURIOUS-outside-anatomy (orange). Each bucket drawn on its OWN panel."""
    picks = [r for r in results if r.image_path and os.path.isfile(r.image_path)
             and (r.causal_patches or r.spurious_patches)][:n_examples]
    if not picks:
        print("  [bbox] no drawable results."); return
    fig, axes = plt.subplots(len(picks), 3, figsize=(11, 3.6 * len(picks)))
    if len(picks) == 1: axes = np.array([axes])
    titles = ["causal", "spurious in-anatomy", "spurious outside-anatomy"]
    for ridx, r in enumerate(picks):
        pil = Image.open(r.image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
        for c in range(3):
            axes[ridx, c].imshow(pil); axes[ridx, c].axis("off")
            if ridx == 0: axes[ridx, c].set_title(titles[c], fontsize=11)
        for p in r.causal_patches:
            col = "#1e8cff" if p.selection_source == "fallback" else "#00dc3c"
            x1, y1, x2, y2 = p.box
            axes[ridx, 0].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                    fill=False, edgecolor=col, lw=2,
                                    linestyle="--" if p.selection_source == "fallback" else "-"))
        for p in r.spurious_patches:
            if p.spurious_source == "in_anatomy":
                x1, y1, x2, y2 = p.box
                axes[ridx, 1].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                        fill=False, edgecolor="#dc1e1e", lw=2))
            else:
                x1, y1, x2, y2 = p.box
                axes[ridx, 2].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                        fill=False, edgecolor="#ffa500", lw=2, linestyle=":"))
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=140, bbox_inches="tight"); print(f"Saved: {save_path}")
    plt.show()

print("[cell 6] discovery, A* localization/image metrics, GradCAM grid, bbox-by-bucket ready.")


In [ ]:
# =============================================================================
# CELL 7 / 8  —  RUN FLAGS  +  INTERNAL SMOKE TEST  +  STAGE 2+3 DRIVER
# =============================================================================
#   SECTION 7.1  execution flags
#   SECTION 7.2  run_internal_smoke_test() — validates every fix with NO GPU,
#                NO CheXzero checkpoint and NO dataset (synthetic tensors only)
#   SECTION 7.3  Stage 2 + Stage 3 real driver (needs data + checkpoint)
#   SECTION 7.4  A* metric + figure reporting after Stage 3
# -----------------------------------------------------------------------------

# ==================== SECTION 7.1 — flags =================================
RUN_SMOKE_TEST = True     # internal self-test (safe anywhere; ~2s)
RUN_STAGE23    = True      # Stage 2 (query plans) + Stage 3 (patch discovery)
RUN_ABLATIONS  = False     # cell 8 — needs RUN_STAGE23 objects in memory

# ==================== SECTION 7.2 — internal smoke test ===================
def run_internal_smoke_test():
    """Self-contained validation of the v14 fixes. Uses random tensors and a
    tiny stub text/image encoder — no CUDA, no CheXzero, no data required."""
    print("\n" + "=" * 70 + "\nINTERNAL SMOKE TEST (v14 fixes)\n" + "=" * 70)
    dev = torch.device("cpu"); ok = True
    ls = torch.tensor(100.0)

    # -- FIX A: binary prompts decouple classes; 5-way softmax couples them ----
    torch.manual_seed(0)
    pos = F.normalize(torch.randn(N_CLASSES, CLIP_EMBED_DIM), dim=-1)
    neg = F.normalize(torch.randn(N_CLASSES, CLIP_EMBED_DIM), dim=-1)
    emb = F.normalize(torch.randn(4, CLIP_EMBED_DIM), dim=-1)
    bp = binary_class_probs_gpu(emb, pos, neg, ls)
    assert bp.shape == (4, N_CLASSES), "binary prob shape"
    assert torch.all((bp >= 0) & (bp <= 1)), "binary prob range"
    # perturbing OTHER classes' prompts must NOT change target-class prob (decoupled)
    pos2 = pos.clone(); pos2[1:] = F.normalize(torch.randn(N_CLASSES - 1, CLIP_EMBED_DIM), dim=-1)
    bp2 = binary_class_probs_gpu(emb, pos2, neg, ls)
    decoupled = torch.allclose(bp[:, 0], bp2[:, 0], atol=1e-6)
    five = zeroshot_probs_gpu(emb, pos, ls)  # 5-way for contrast
    print(f"  [FIX A] binary decoupled across classes: {decoupled}  "
          f"(5-way softmax row sums to 1: {torch.allclose(five.sum(-1), torch.ones(4))})")
    ok &= bool(decoupled)

    # -- FIX B: global ensemble arithmetic ------------------------------------
    patch_prob = torch.tensor([0.2, 0.8, 0.5]); gprob = 0.9; a = 0.5
    ens = a * gprob + (1 - a) * patch_prob
    assert torch.allclose(ens, torch.tensor([0.55, 0.85, 0.70])), "ensemble math"
    print(f"  [FIX B] ensemble alpha={a}: {ens.tolist()}  (mixes global {gprob} with patch)")

    # -- FIX C: per-class scale map keeps 64px for small findings -------------
    keeps64 = all(64 in _SCALE_MAP[c] for c in ("Atelectasis", "Edema"))
    print(f"  [FIX C] Atelectasis/Edema keep 64px scale: {keeps64}")
    ok &= keeps64

    # -- Box geometry: always square, in-bounds, >= MIN_BOX_SIZE --------------
    for (x, y, s) in [(500, 500, 128), (-10, -10, 8), (0, 0, 9999)]:
        b = _clamp_box_square(x, y, s)
        if b is not None:
            w = b[2] - b[0]; h = b[3] - b[1]
            assert w == h and w >= MIN_BOX_SIZE and 0 <= b[0] and b[2] <= IMAGE_SIZE, f"geom {b}"
    hm = torch.zeros(IMAGE_SIZE, IMAGE_SIZE); hm[100:300, 100:300] = 1.0
    boxes = extract_candidate_boxes_gpu(hm, 128, 32, 0.5)
    print(f"  [geom] extracted {len(boxes)} square in-bounds boxes from synthetic heatmap")
    ok &= len(boxes) > 0

    # -- Persistence mask on a synthetic 2-blob GradCAM -----------------------
    g = torch.zeros(IMAGE_SIZE, IMAGE_SIZE)
    yy, xx = torch.meshgrid(torch.arange(IMAGE_SIZE), torch.arange(IMAGE_SIZE), indexing="ij")
    g += torch.exp(-(((xx - 150.0) ** 2 + (yy - 150.0) ** 2) / (2 * 40.0 ** 2)))
    g += 0.8 * torch.exp(-(((xx - 380.0) ** 2 + (yy - 360.0) ** 2) / (2 * 30.0 ** 2)))
    m, peaks = compute_persistent_causal_mask(g)
    print(f"  [persist] mask found: {m is not None} | peaks={len(peaks)} (expect >=1)")
    ok &= (m is not None)

    # -- FIX (branch): RandomState.integers bug is gone; default_rng works ----
    def _rand_boxes(n, rng):
        out = []
        for _ in range(n):
            side = int(rng.choice([64, 128, 256])); side = min(side, IMAGE_SIZE)
            x1 = int(rng.integers(0, max(1, IMAGE_SIZE - side)))
            y1 = int(rng.integers(0, max(1, IMAGE_SIZE - side)))
            out.append((x1, y1, x1 + side, y1 + side))
        return out
    rb = _rand_boxes(8, np.random.default_rng(42 + 991))
    legacy_ok = False
    try:
        _rand_boxes(1, np.random.RandomState(0))    # must raise (proves old bug)
    except AttributeError:
        legacy_ok = True
    print(f"  [RNG] default_rng random boxes OK ({len(rb)}) | legacy RandomState correctly rejected: {legacy_ok}")
    ok &= (len(rb) == 8 and legacy_ok)

    # -- A* localization math: pointing game + IoU ----------------------------
    gt = (100, 100, 300, 300)
    assert _center_in((120, 120, 280, 280), gt) and not _center_in((350, 350, 400, 400), gt)
    assert _iou((100, 100, 300, 300), gt) == 1.0 and _iou((350, 350, 400, 400), gt) == 0.0
    print("  [loc] pointing-game + IoU math verified")

    # -- A* image metric: AUPRC vs prevalence ---------------------------------
    yt = np.array([1, 0, 0, 1, 0, 0, 0, 1, 0, 0]); ys = np.array([.9, .1, .2, .8, .3, .1, .05, .7, .2, .1])
    print(f"  [metric] AUROC={roc_auc_score(yt, ys):.3f} AUPRC={average_precision_score(yt, ys):.3f} "
          f"prevalence={yt.mean():.3f}")

    print("=" * 70)
    print("SMOKE TEST: " + ("ALL CHECKS PASSED" if ok else "FAILURES - inspect above"))
    print("=" * 70)
    return ok

if RUN_SMOKE_TEST:
    _smoke_ok = run_internal_smoke_test()

# ==================== SECTION 7.3 — Stage 2 + 3 driver ====================
if RUN_STAGE23:
    print(f"\nDevice: {Device}")
    if USE_CUDA:
        print(f"GPU : {torch.cuda.get_device_name(0)} | "
              f"VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    if DATASET == "NIH":
        print("\n" + "=" * 70 + "\nREADING CSV\n" + "=" * 70)
        train_df = read_pairs_csv(TRAIN_CSV); val_df = read_pairs_csv(VAL_CSV); test_df = read_pairs_csv(TEST_CSV)
        print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
        _view_default = "PA"; _view_from_report = False
    else:
        print("\n" + "=" * 70 + "\nREADING MIMIC-CXR COMBINED CSV\n" + "=" * 70)
        assert os.path.exists(COMBINED_CSV), f"CSV not found: {COMBINED_CSV}"
        full_df = pd.read_csv(COMBINED_CSV)
        def _norm_split(s):
            s = str(s).strip().lower()
            if s in ("train", "training"): return "train"
            if s in ("validate", "valid", "val", "validation", "dev"): return "val"
            if s in ("test", "testing", "eval"): return "test"
            return s
        full_df["_split"] = full_df["split"].apply(_norm_split) if "split" in full_df.columns else "train"
        train_df = full_df[full_df["_split"] == "train"].reset_index(drop=True)
        val_df = full_df[full_df["_split"] == "val"].reset_index(drop=True)
        test_n = int((full_df["_split"] == "test").sum()); test_df = None
        if len(val_df) == 0 and len(train_df) > 0:
            n_val = max(1, int(len(train_df) * VAL_CARVE_FRAC))
            val_df = train_df.sample(n_val, random_state=SEED)
            train_df = train_df.drop(val_df.index).reset_index(drop=True); val_df = val_df.reset_index(drop=True)
        _view_default = "AUTO"; _view_from_report = True

    if MULTICLASS_ONLY:
        train_df = filter_multiclass_rows(train_df, "Train"); val_df = filter_multiclass_rows(val_df, "Val")
    if SMOKE_TEST:
        train_df = train_df.sample(min(len(train_df), SMOKE_TRAIN_N), random_state=SEED).reset_index(drop=True)
        val_df = val_df.sample(min(len(val_df), SMOKE_VAL_N), random_state=SEED).reset_index(drop=True)
        print(f"SMOKE_TEST(real data): train={len(train_df)} val={len(val_df)}")
    else:
        if MAX_TRAIN_IMAGES is not None and len(train_df) > MAX_TRAIN_IMAGES:
            train_df = train_df.sample(MAX_TRAIN_IMAGES, random_state=SEED).reset_index(drop=True)
        if MAX_VAL_IMAGES is not None and len(val_df) > MAX_VAL_IMAGES:
            val_df = val_df.sample(MAX_VAL_IMAGES, random_state=SEED).reset_index(drop=True)

    print(f"Processing - Train: {len(train_df):,} | Val: {len(val_df):,}")

    print("\n" + "=" * 70 + "\nSTAGE 2 - RULE-BASED QUERY PLANS\n" + "=" * 70)
    VOCAB = mine_vocab_from_reports(train_df)
    train_plans = run_stage2(train_df, VOCAB, MAX_TRAIN_IMAGES if not SMOKE_TEST else None,
                             "Stage 2 - train", USE_LABEL_BACKUP_TRAIN,
                             view_default=_view_default, view_from_report=_view_from_report)
    val_plans = run_stage2(val_df, VOCAB, None, "Stage 2 - val", USE_LABEL_BACKUP_VAL,
                           view_default=_view_default, view_from_report=_view_from_report)
    split_stats(train_plans, "Train"); split_stats(val_plans, "Val")
    with open(STAGE2_OUT, "wb") as f:
        pickle.dump({"train": train_plans, "val": val_plans, "vocab": VOCAB}, f, protocol=4)

    print("\n" + "=" * 70 + "\nSTAGE 3 - PATCH DISCOVERY (binary prompts + global ensemble)\n" + "=" * 70)
    chexzero, cz_prep = load_chexzero(CHEXZERO_CKPT, Device)
    ctm_gpu, ls_gpu = build_class_text_matrix(chexzero, Device)
    POS_MAT, NEG_MAT = build_binary_prompt_matrices(chexzero, Device)   # [v14 FIX A]
    train_plans = fill_query_vectors(train_plans, chexzero, Device, ctm=ctm_gpu)
    val_plans = fill_query_vectors(val_plans, chexzero, Device, ctm=ctm_gpu)
    gradcam = ViTGradCAM(chexzero, cz_prep, Device, ctm_gpu, ls_gpu)

    train_results, val_results = [], []; train_completed = val_completed = False
    try:
        train_results, train_completed, _ = run_stage3(
            train_plans, "train", gradcam, chexzero, cz_prep, Device, ctm_gpu, ls_gpu,
            deadline=_DEADLINE, out_dir=OUT_DIR, pos_mat=POS_MAT, neg_mat=NEG_MAT)
        if train_completed and _budget_left_h(_DEADLINE) > 0.15:
            val_results, val_completed, _ = run_stage3(
                val_plans, "val", gradcam, chexzero, cz_prep, Device, ctm_gpu, ls_gpu,
                deadline=_DEADLINE, out_dir=OUT_DIR, pos_mat=POS_MAT, neg_mat=NEG_MAT)
        else:
            print("Skipping val this session - re-run to resume.")
    finally:
        gradcam.remove_hooks()

    rd = {"train": train_results, "val": val_results}
    print("\n" + "=" * 70 + "\nSTAGE 3 SUMMARY\n" + "=" * 70)
    summarize_stage3(train_results, "Train")
    if val_results: summarize_stage3(val_results, "Val")
    print(f"\nCounters: {dict(_COUNTERS)}")

    with open(STAGE3_RESULTS, "wb") as f: pickle.dump(rd, f, protocol=4)
    cp, sp, op = split_embeddings_three_ways(rd)
    torch.save(cp, STAGE3_CAUSAL_PT); torch.save(sp, STAGE3_SPIN_PT); torch.save(op, STAGE3_SPOUT_PT)
    with open(STAGE3_META, "wb") as f:
        pickle.dump({"version": VERSION, "dataset": DATASET_NAME, "target_classes": TARGET_CLASSES,
                     "use_binary_prompts": USE_BINARY_PROMPTS, "global_ensemble_alpha": GLOBAL_ENSEMBLE_ALPHA,
                     "allow_fallback": ALLOW_FALLBACK,
                     "counts": {"causal": cp["n"], "spur_in": sp["n"], "spur_out": op["n"]},
                     "counters": dict(_COUNTERS)}, f, protocol=4)
    print(f"Saved causal={cp['n']:,} spur_in={sp['n']:,} spur_out={op['n']:,}")

    # ---------------- SECTION 7.4 — A* metrics + must-have figures ----------
    plans_by_name = {p.image_name: p for p in (train_plans + val_plans)}
    GT_LUT = build_gt_lookup(train_df, val_df)
    BBOX_GT = load_bbox_gt(BBOX_CSV)
    image_level_metrics(rd, GT_LUT)
    evaluate_localization(rd, BBOX_GT)
    _live_gc = ViTGradCAM(chexzero, cz_prep, Device, ctm_gpu, ls_gpu)
    try:
        plot_causal_gradcam_grid(train_results + val_results, plans_by_name, _live_gc,
                                 save_path=f"{OUT_DIR}/causal_gradcam_grid.png")
    finally:
        _live_gc.remove_hooks()
    plot_bboxes_by_bucket(train_results + val_results, n_examples=4,
                          save_path=f"{OUT_DIR}/bboxes_by_bucket.png")
    print("\nStage 3 + A* reporting complete.")


In [ ]:
# =============================================================================
# CELL 8 / 8  —  PATCH-MINING ABLATIONS  +  NEGATIVE CONTROLS  +  A* REPORT
# =============================================================================
#   SECTION 8.1  ablation config + model/plan guard + binary-prompt guard
#   SECTION 8.2  GT lookup + prior heatmaps + AUROC/AUPRC/CI helpers
#   SECTION 8.3  controls: random boxes / gcam-peak   [RandomState -> default_rng FIX]
#   SECTION 8.4  one-config evaluator (uses binary probs + global ensemble)
#   SECTION 8.5  paired dAUROC-vs-FULL bootstrap
#   SECTION 8.6  apply/restore + grids + main loop + master tables + plots
# The catastrophic bug this branch targets: _control_scores built a legacy
# np.random.RandomState, then _rand_boxes called rng.integers() (Generator API)
# which RandomState lacks -> AttributeError killed the run right after the grid
# arms saved, so controls/summary/plots never produced. Fixed in SECTION 8.3.
# -----------------------------------------------------------------------------
if RUN_ABLATIONS:
    from sklearn.metrics import roc_auc_score, average_precision_score

    # ---------------- SECTION 8.1 — config + guards -----------------------
    ABL_SUBSET_N             = 400        # raise for the final table
    ABL_SEEDS                = [42, 1, 7]  # multi-seed for the paper
    ABL_PRIOR_THR            = 0.10
    ABL_TOPK                 = 3
    ABL_WALLCLOCK_BUDGET_MIN = 180
    ABL_RUN_STABILITY        = True
    ABL_STAB_N               = 25
    ABL_BOOTSTRAP            = 1000
    ABL_PAIRED_BOOT          = 2000
    ABL_CTRL_RANDOM_K        = 8
    ABL_OUT_CSV              = f"{OUT_DIR}/patch_ablations_{VERSION}.csv"
    ABL_PERCLASS_CSV         = f"{OUT_DIR}/patch_ablations_{VERSION}_perclass.csv"
    ABL_PAIRED_CSV           = f"{OUT_DIR}/patch_ablations_{VERSION}_paired.csv"

    _ABL_T0 = time.time()
    def _abl_left_min():
        return ABL_WALLCLOCK_BUDGET_MIN - (time.time() - _ABL_T0) / 60.0

    try:
        chexzero, cz_prep, ctm_gpu, ls_gpu
    except NameError:
        chexzero, cz_prep = load_chexzero(CHEXZERO_CKPT, Device)
        ctm_gpu, ls_gpu   = build_class_text_matrix(chexzero, Device)
        train_plans = fill_query_vectors(train_plans, chexzero, Device, ctm=ctm_gpu)
        val_plans   = fill_query_vectors(val_plans,   chexzero, Device, ctm=ctm_gpu)
    try:
        POS_MAT, NEG_MAT                          # built in cell 7
    except NameError:
        POS_MAT, NEG_MAT = build_binary_prompt_matrices(chexzero, Device)

    # live-tunable persistence override (kept from v13)
    def compute_persistent_causal_mask(gcam_gpu):
        if gcam_gpu is None or not PERSISTENCE_ENABLED: return None, []
        gcam_np = gcam_gpu.cpu().numpy().astype(np.float64)
        if gcam_np.max() - gcam_np.min() < 0.01: return None, []
        mask, peaks = _persistent_mask_scipy(gcam_np, top_k=PERSISTENCE_TOP_K, n_levels=PERSISTENCE_N_LEVELS)
        if mask is None: return None, []
        if CAUSAL_MASK_DILATE_PX > 0:
            mask = binary_dilation(mask, iterations=int(CAUSAL_MASK_DILATE_PX))
        return torch.from_numpy(mask.astype(np.float32)).to(gcam_gpu.device), peaks

    _ALL_PLANS = [p for p in (val_plans + train_plans)
                  if p.query_items and any(qi.query_vector is not None for qi in p.query_items)]
    def _subset_for_seed(seed):
        rng = np.random.RandomState(seed)   # subset selection only (choice) — fine on RandomState
        if len(_ALL_PLANS) > ABL_SUBSET_N:
            idx = rng.choice(len(_ALL_PLANS), ABL_SUBSET_N, replace=False)
            return [_ALL_PLANS[i] for i in sorted(idx)]
        return list(_ALL_PLANS)
    print(f"Ablation pool: {len(_ALL_PLANS)} plans | seeds={ABL_SEEDS} | n={ABL_SUBSET_N}")

    # ---------------- SECTION 8.2 — GT + metric helpers -------------------
    GT_LUT = build_gt_lookup(train_df, val_df)   # canonical lookup from cell 6
    print(f"GT label lookup: {len(GT_LUT):,} images")

    def _class_prior_hmaps(plan):
        hm = {}
        for qi in plan.query_items:
            if qi.anatomical_prior and qi.pathology in TARGET_CLASSES and qi.pathology not in hm:
                p = qi.anatomical_prior
                h = make_gaussian_heatmap_gpu(p.center_x, p.center_y, p.sigma_x, p.sigma_y,
                                              IMAGE_SIZE, IMAGE_SIZE, Device)
                mx = h.max(); hm[qi.pathology] = (h / mx) if mx > 0 else h
        return hm

    def _auroc(yt, ys):
        yt, ys = np.asarray(yt), np.asarray(ys)
        if len(np.unique(yt)) != 2:
            return float("nan"), int((yt == 1).sum()), int((yt == 0).sum())
        try:
            return float(roc_auc_score(yt, ys)), int((yt == 1).sum()), int((yt == 0).sum())
        except Exception:
            return float("nan"), int((yt == 1).sum()), int((yt == 0).sum())
    def _auprc(yt, ys):
        yt, ys = np.asarray(yt), np.asarray(ys)
        if len(np.unique(yt)) != 2: return float("nan")
        try: return float(average_precision_score(yt, ys))
        except Exception: return float("nan")
    def _auroc_ci(yt, ys, n_boot=ABL_BOOTSTRAP, seed=0):
        yt, ys = np.asarray(yt), np.asarray(ys)
        if len(np.unique(yt)) != 2: return (float("nan"), float("nan"))
        r = np.random.default_rng(seed); vals = []; n = len(yt)
        for _ in range(n_boot):
            i = r.integers(0, n, n)
            if len(np.unique(yt[i])) == 2:
                try: vals.append(roc_auc_score(yt[i], ys[i]))
                except Exception: pass
        if not vals: return (float("nan"), float("nan"))
        return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))

    # ---------------- SECTION 8.3 — controls  [THE RandomState FIX] --------
    def _rand_boxes(n, rng):
        out = []
        for _ in range(n):
            side = int(rng.choice([64, 128, 256])); side = min(side, IMAGE_SIZE)
            x1 = int(rng.integers(0, max(1, IMAGE_SIZE - side)))
            y1 = int(rng.integers(0, max(1, IMAGE_SIZE - side)))
            out.append((x1, y1, x1 + side, y1 + side))
        return out
    def _peak_box(hmap_gpu, side=128):
        idx = int(torch.argmax(hmap_gpu).item()); H = hmap_gpu.shape[-1]
        cy, cx = idx // H, idx % H
        return _clamp_box_square(cx - side // 2, cy - side // 2, side)

    def _control_scores(plans, mode, gradcam, seed):
        # [FIX] default_rng exposes .integers used by _rand_boxes. The old code used
        # np.random.RandomState(seed+991) (legacy API, only .randint) -> AttributeError.
        rng = np.random.default_rng(seed + 991)
        scores = {c: [] for c in TARGET_CLASSES}; gts = {c: [] for c in TARGET_CLASSES}
        for plan in plans:
            gt = GT_LUT.get(os.path.basename(str(plan.image_name)))
            if gt is None: continue
            try:
                pil = Image.open(plan.image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
            except Exception:
                continue
            comp = get_composite_heatmap_gpu(plan, IMAGE_SIZE, IMAGE_SIZE, Device)
            queried = {qi.pathology: qi for qi in plan.query_items
                       if qi.pathology in TARGET_CLASSES and not qi.negated and qi.query_vector is not None}
            best = {c: 0.0 for c in TARGET_CLASSES}
            if mode == "random":
                boxes = [b for b in _rand_boxes(ABL_CTRL_RANDOM_K, rng) if b is not None]
                if boxes:
                    embs = encode_patch_batch_gpu([pil.crop(b) for b in boxes], chexzero, cz_prep, Device)
                    probs = class_probs_gpu(embs, ctm_gpu, ls_gpu, POS_MAT, NEG_MAT)
                    for c in TARGET_CLASSES:
                        ti = TARGET_CLASSES.index(c); best[c] = float(probs[:, ti].max().item())
            else:  # gcam_peak
                for c, qi in queried.items():
                    ti = TARGET_CLASSES.index(c)
                    gcam = gradcam.compute_combined(pil, qi.query_vector, ti) if ENABLE_GRADCAM else None
                    hmap = gcam if gcam is not None else comp; b = _peak_box(hmap)
                    if b is None: continue
                    embs = encode_patch_batch_gpu([pil.crop(b)], chexzero, cz_prep, Device)
                    probs = class_probs_gpu(embs, ctm_gpu, ls_gpu, POS_MAT, NEG_MAT)
                    best[c] = float(probs[0, ti].item())
            for c in TARGET_CLASSES:
                lv = gt.get(c, float("nan"))
                if lv == lv: scores[c].append(best[c]); gts[c].append(lv)
        return scores, gts

    # ---------------- SECTION 8.4 — one-config evaluator ------------------
    def _eval_current_config(plans, gradcam, label="", with_ci=False, seed=0):
        t0 = time.time(); n_img = len(plans)
        n_causal = n_true = n_fb = n_img_causal = n_in_prior = n_leak = 0
        sims = []; n_spin = 0; n_spout = 0; band_hits = 0; band_tot = 0
        scores_max = {c: [] for c in TARGET_CLASSES}; scores_topk = {c: [] for c in TARGET_CLASSES}
        gts = {c: [] for c in TARGET_CLASSES}
        pc = {c: dict(causal=0, true=0, fb=0, prior=0, leak=0, spin=0, img=0, queried=0, sims=[])
              for c in TARGET_CLASSES}
        for plan in plans:
            res = discover_patches_for_plan(plan, gradcam, chexzero, cz_prep, Device, ctm_gpu, ls_gpu,
                                            "abl", pos_mat=POS_MAT, neg_mat=NEG_MAT)
            cps = res.causal_patches
            chm = get_composite_heatmap_gpu(plan, IMAGE_SIZE, IMAGE_SIZE, Device)
            cpr = _class_prior_hmaps(plan)
            queried = {qi.pathology for qi in plan.query_items
                       if qi.pathology in TARGET_CLASSES and not qi.negated}
            for c in queried: pc[c]["queried"] += 1
            if cps: n_img_causal += 1
            n_causal += len(cps); imgs_c = set()
            for p in cps:
                sims.append(p.semantic_score); band_tot += 1
                if 0.24 <= p.semantic_score <= 0.34: band_hits += 1
                if p.selection_source == "fallback": n_fb += 1
                else: n_true += 1
                cx = min((p.box[0] + p.box[2]) // 2, IMAGE_SIZE - 1)
                cy = min((p.box[1] + p.box[3]) // 2, IMAGE_SIZE - 1)
                if float(chm[cy, cx].item()) >= ABL_PRIOR_THR: n_in_prior += 1
                else: n_leak += 1
                c = p.pathology
                if c in pc:
                    pc[c]["causal"] += 1
                    pc[c]["fb" if p.selection_source == "fallback" else "true"] += 1
                    pc[c]["sims"].append(p.semantic_score)
                    hmc = cpr.get(c)
                    if hmc is not None and float(hmc[cy, cx].item()) >= ABL_PRIOR_THR: pc[c]["prior"] += 1
                    else: pc[c]["leak"] += 1
                    imgs_c.add(c)
            for c in imgs_c: pc[c]["img"] += 1
            for p in res.spurious_patches:
                if p.spurious_source == "outside_anatomy": n_spout += 1
                else:
                    n_spin += 1
                    if p.pathology in pc: pc[p.pathology]["spin"] += 1
            gt = GT_LUT.get(os.path.basename(str(plan.image_name)))
            if gt is not None:
                probs_by_c = {c: [] for c in TARGET_CLASSES}
                for p in cps:
                    if p.pathology in probs_by_c: probs_by_c[p.pathology].append(float(p.zeroshot_prob))
                for c in TARGET_CLASSES:
                    lv = gt.get(c, float("nan"))
                    if lv == lv:
                        pr = sorted(probs_by_c[c], reverse=True)
                        scores_max[c].append(pr[0] if pr else 0.0)
                        scores_topk[c].append(float(np.mean(pr[:ABL_TOPK])) if pr else 0.0)
                        gts[c].append(lv)
        pc_rows = []
        for c in TARGET_CLASSES:
            au, npos, nneg = _auroc(gts[c], scores_max[c]); au_tk, _, _ = _auroc(gts[c], scores_topk[c])
            ap = _auprc(gts[c], scores_max[c])
            lo, hi = _auroc_ci(gts[c], scores_max[c], seed=seed) if with_ci else (float("nan"), float("nan"))
            q = max(pc[c]["queried"], 1); ncz = max(pc[c]["causal"], 1)
            pc_rows.append({"label": label, "seed": seed, "class": c, "auroc": au, "auroc_topk": au_tk,
                            "auprc": ap, "ci_lo": lo, "ci_hi": hi, "support_pos": npos, "support_neg": nneg,
                            "causal": pc[c]["causal"], "causal_per_qimg": pc[c]["causal"]/q,
                            "pct_qimg_causal": 100.0*pc[c]["img"]/q, "abstain_pct": 100.0*(1.0 - pc[c]["img"]/q),
                            "true_pct": 100.0*pc[c]["true"]/ncz, "fallback_pct": 100.0*pc[c]["fb"]/ncz,
                            "prior_pct": 100.0*pc[c]["prior"]/ncz, "leak_pct": 100.0*pc[c]["leak"]/ncz,
                            "mean_sim": float(np.mean(pc[c]["sims"])) if pc[c]["sims"] else 0.0,
                            "spur_in": pc[c]["spin"]})
        macro = float(np.nanmean([r["auroc"] for r in pc_rows]))
        macro_tk = float(np.nanmean([r["auroc_topk"] for r in pc_rows]))
        macro_ap = float(np.nanmean([r["auprc"] for r in pc_rows]))
        overall = {"label": label, "seed": seed, "causal_per_img": n_causal / max(n_img, 1),
                   "pct_img_causal": 100.0 * n_img_causal / max(n_img, 1),
                   "abstain_pct": 100.0 * (1.0 - n_img_causal / max(n_img, 1)),
                   "true_causal_pct": 100.0 * n_true / max(n_causal, 1),
                   "fallback_pct": 100.0 * n_fb / max(n_causal, 1),
                   "prior_overlap_pct": 100.0 * n_in_prior / max(n_causal, 1),
                   "leak_pct": 100.0 * n_leak / max(n_causal, 1),
                   "band_occ_pct": 100.0 * band_hits / max(band_tot, 1),
                   "mean_sim": float(np.mean(sims)) if sims else 0.0, "spur_in": n_spin, "spur_out": n_spout,
                   "macroAUROC": macro, "macroAUROC_topk": macro_tk, "macroAUPRC": macro_ap,
                   "n_auc_classes": int(np.sum(~np.isnan([r["auroc"] for r in pc_rows]))),
                   "sec_img": (time.time() - t0) / max(n_img, 1)}
        raw = {"scores_max": scores_max, "scores_topk": scores_topk, "gts": gts}
        return overall, pc_rows, raw

    # ---------------- SECTION 8.5 — paired dAUROC bootstrap ----------------
    def _paired_delta_vs_full(raw_cfg, raw_full, n_boot=ABL_PAIRED_BOOT, seed=0):
        r = np.random.default_rng(seed + 4242); deltas = []; cls_arrays = []
        for c in TARGET_CLASSES:
            g = np.asarray(raw_full["gts"][c]); sc = np.asarray(raw_cfg["scores_max"][c]); sf = np.asarray(raw_full["scores_max"][c])
            if len(g) == len(sc) == len(sf) and len(np.unique(g)) == 2: cls_arrays.append((g, sc, sf))
        if not cls_arrays: return (float("nan"),)*4
        for _ in range(n_boot):
            per_cls = []
            for g, sc, sf in cls_arrays:
                n = len(g); i = r.integers(0, n, n)
                if len(np.unique(g[i])) != 2: continue
                try: per_cls.append(roc_auc_score(g[i], sc[i]) - roc_auc_score(g[i], sf[i]))
                except Exception: pass
            if per_cls: deltas.append(float(np.mean(per_cls)))
        if not deltas: return (float("nan"),)*4
        deltas = np.asarray(deltas)
        return (float(deltas.mean()), float(np.percentile(deltas, 2.5)),
                float(np.percentile(deltas, 97.5)), float(np.mean(deltas <= 0.0)))

    # ---------------- SECTION 8.6 — apply/restore + grids + main loop ------
    _ABL_KEYS = ["CAUSAL_CONTAIN_MODE", "CAUSAL_MASK_COVER_FRAC", "CAUSAL_MASK_DILATE_PX",
                 "PERSISTENCE_TOP_K", "PERSISTENCE_N_LEVELS", "SEMANTIC_THRESHOLD_PER_CLASS",
                 "SEMANTIC_THRESHOLD", "ALLOW_FALLBACK", "GLOBAL_ENSEMBLE_ALPHA", "USE_BINARY_PROMPTS"]
    _BASE = {k: copy.deepcopy(globals()[k]) for k in _ABL_KEYS}
    def _apply(ov):
        for k in _ABL_KEYS: globals()[k] = copy.deepcopy(_BASE[k])
        if "sem_scale" in ov:
            s = ov["sem_scale"]
            globals()["SEMANTIC_THRESHOLD_PER_CLASS"] = {c: v*s for c, v in _BASE["SEMANTIC_THRESHOLD_PER_CLASS"].items()}
            globals()["SEMANTIC_THRESHOLD"] = _BASE["SEMANTIC_THRESHOLD"] * s
        if "sem_abs" in ov:
            a = ov["sem_abs"]
            globals()["SEMANTIC_THRESHOLD_PER_CLASS"] = {c: a for c in _BASE["SEMANTIC_THRESHOLD_PER_CLASS"]}
            globals()["SEMANTIC_THRESHOLD"] = a
        for k, v in ov.items():
            if k in ("sem_scale", "sem_abs"): continue
            globals()[k] = v
    def _restore():
        for k in _ABL_KEYS: globals()[k] = copy.deepcopy(_BASE[k])

    CONFIGS = [
        ("FULL",                {}),
        ("A:contain=center",    {"CAUSAL_CONTAIN_MODE": "center"}),
        ("A:contain=peak",      {"CAUSAL_CONTAIN_MODE": "peak"}),
        ("A:contain=cover",     {"CAUSAL_CONTAIN_MODE": "cover"}),
        ("B:topK=1",            {"PERSISTENCE_TOP_K": 1}),
        ("B:topK=3",            {"PERSISTENCE_TOP_K": 3}),
        ("C:levels=16",         {"PERSISTENCE_N_LEVELS": 16}),
        ("C:levels=48",         {"PERSISTENCE_N_LEVELS": 48}),
        ("D:dilate=0",          {"CAUSAL_MASK_DILATE_PX": 0}),
        ("D:dilate=12",         {"CAUSAL_MASK_DILATE_PX": 12}),
        ("E:cover=0.20",        {"CAUSAL_MASK_COVER_FRAC": 0.20}),
        ("E:cover=0.50",        {"CAUSAL_MASK_COVER_FRAC": 0.50}),
        ("F:sem_abs=0.24",      {"sem_abs": 0.24}),
        ("F:sem_abs=0.30",      {"sem_abs": 0.30}),
        ("F:sem_abs=0.34",      {"sem_abs": 0.34}),
        ("G:no_fallback",       {"ALLOW_FALLBACK": False}),
        # [v14] scoring-model ablations — the two catastrophic-output fixes
        ("H:5way_softmax",      {"USE_BINARY_PROMPTS": False}),
        ("I:ensemble=0.0",      {"GLOBAL_ENSEMBLE_ALPHA": 0.0}),
        ("I:ensemble=1.0",      {"GLOBAL_ENSEMBLE_ALPHA": 1.0}),
    ]
    CONTROLS = ["CTRL:random", "CTRL:gcam_peak"]
    _ORDER = [lab for lab, _ in CONFIGS] + CONTROLS

    ROWS, done = [], set()
    if os.path.exists(ABL_OUT_CSV):
        prev = pd.read_csv(ABL_OUT_CSV); ROWS = prev.to_dict("records")
        done = set(prev["label"].astype(str) + "|" + prev["seed"].astype(str))
        print(f"Resuming: {len(done)} (config,seed) cells already saved")
    PC_ROWS = pd.read_csv(ABL_PERCLASS_CSV).to_dict("records") if os.path.exists(ABL_PERCLASS_CSV) else []
    PAIRED_ROWS = pd.read_csv(ABL_PAIRED_CSV).to_dict("records") if os.path.exists(ABL_PAIRED_CSV) else []
    def _save():
        for rows, path in [(ROWS, ABL_OUT_CSV), (PC_ROWS, ABL_PERCLASS_CSV), (PAIRED_ROWS, ABL_PAIRED_CSV)]:
            tmp = path + ".tmp"; pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, path)

    _stopped = False; FULL_AUROC = float("nan"); full_pc = []
    for seed in ABL_SEEDS:
        if _stopped: break
        np.random.seed(seed); PLANS = _subset_for_seed(seed)
        print(f"\n### SEED {seed} - {len(PLANS)} plans ###")
        gradcam = ViTGradCAM(chexzero, cz_prep, Device, ctm_gpu, ls_gpu); raw_full = None
        try:
            key = f"FULL|{seed}"
            if key not in done:
                _apply({}); r0, pc0, raw_full = _eval_current_config(PLANS, gradcam, "FULL", with_ci=True, seed=seed)
                ROWS.append(r0); PC_ROWS += pc0; done.add(key); _save()
                if seed == ABL_SEEDS[0]: FULL_AUROC = float(r0["macroAUROC"]); full_pc = pc0
                print(f"FULL[{seed}] AUROC(max)={r0['macroAUROC']:.4f} AUPRC={r0['macroAUPRC']:.4f} "
                      f"| band_occ={r0['band_occ_pct']:.1f}% | abstain={r0['abstain_pct']:.1f}%")
            else:
                _apply({}); _, _, raw_full = _eval_current_config(PLANS, gradcam, "FULL", with_ci=False, seed=seed)
                if seed == ABL_SEEDS[0]:
                    r0 = next(r for r in ROWS if str(r["label"]) == "FULL" and int(r["seed"]) == seed)
                    FULL_AUROC = float(r0["macroAUROC"])
                    full_pc = [r for r in PC_ROWS if str(r["label"]) == "FULL" and int(r["seed"]) == seed]
            for lab, ov in CONFIGS:
                if lab == "FULL": continue
                key = f"{lab}|{seed}"
                if key in done: continue
                if _abl_left_min() <= 0:
                    print(f"\nBudget reached before '{lab}' (seed {seed})."); _stopped = True; break
                _apply(ov)
                try:
                    r, pcr, raw_c = _eval_current_config(PLANS, gradcam, lab, with_ci=False, seed=seed)
                finally:
                    _restore()
                dmean, dlo, dhi, p_le0 = _paired_delta_vs_full(raw_c, raw_full, seed=seed)
                PAIRED_ROWS.append({"label": lab, "seed": seed, "d_macroAUROC": dmean,
                                    "d_lo": dlo, "d_hi": dhi, "p_delta_le0": p_le0})
                ROWS.append(r); PC_ROWS += pcr; done.add(key)
                sig = "" if (dlo != dlo) else ("  *" if (dlo > 0 or dhi < 0) else "  ns")
                print(f"  {lab:<20} AUROC={r['macroAUROC']:.4f} dvsFULL={dmean:+.4f} "
                      f"[{dlo:+.4f},{dhi:+.4f}] p(<=0)={p_le0:.3f}{sig} | abstain={r['abstain_pct']:.1f}%")
                _save()
            for ctrl in CONTROLS:
                key = f"{ctrl}|{seed}"
                if key in done or _stopped: continue
                mode = "random" if ctrl.endswith("random") else "gcam_peak"
                cs, cg = _control_scores(PLANS, mode, gradcam, seed); rows_c = []
                for c in TARGET_CLASSES:
                    au, npos, nneg = _auroc(cg[c], cs[c]); ap = _auprc(cg[c], cs[c])
                    rows_c.append({"label": ctrl, "seed": seed, "class": c, "auroc": au,
                                   "auroc_topk": float("nan"), "auprc": ap, "ci_lo": float("nan"),
                                   "ci_hi": float("nan"), "support_pos": npos, "support_neg": nneg,
                                   "causal": 0, "causal_per_qimg": 0.0, "pct_qimg_causal": 0.0,
                                   "abstain_pct": float("nan"), "true_pct": float("nan"),
                                   "fallback_pct": float("nan"), "prior_pct": float("nan"),
                                   "leak_pct": float("nan"), "mean_sim": 0.0, "spur_in": 0})
                macro = float(np.nanmean([r["auroc"] for r in rows_c]))
                macro_ap = float(np.nanmean([r["auprc"] for r in rows_c]))
                ROWS.append({"label": ctrl, "seed": seed, "macroAUROC": macro, "macroAUROC_topk": float("nan"),
                             "macroAUPRC": macro_ap, "causal_per_img": 0.0, "pct_img_causal": 0.0,
                             "abstain_pct": float("nan"), "true_causal_pct": float("nan"),
                             "fallback_pct": float("nan"), "prior_overlap_pct": float("nan"),
                             "leak_pct": float("nan"), "band_occ_pct": float("nan"), "mean_sim": 0.0,
                             "spur_in": 0, "spur_out": 0, "n_auc_classes": len(TARGET_CLASSES), "sec_img": 0.0})
                PC_ROWS += rows_c; done.add(key)
                print(f"  {ctrl:<20} macroAUROC={macro:.4f} macroAUPRC={macro_ap:.4f}  [NEGATIVE CONTROL]")
                _save()
        finally:
            gradcam.remove_hooks(); _restore()

    # master tables + plots
    dfR = pd.DataFrame(ROWS)
    print("\n" + "=" * 118)
    print(f"{'config':<20}{'AUROCmax':>9}{'AUROCtk':>9}{'AUPRC':>8}{'true%':>7}{'fb%':>6}"
          f"{'abst%':>7}{'prior%':>8}{'leak%':>7}{'band%':>7}{'sim':>6}")
    print("=" * 118)
    for lab in _ORDER:
        sub = dfR[dfR["label"].astype(str) == lab]
        if sub.empty: continue
        def _m(col):
            return float(np.nanmean(pd.to_numeric(sub[col], errors="coerce"))) if col in sub else float("nan")
        print(f"{lab:<20}{_m('macroAUROC'):>9.4f}{_m('macroAUROC_topk'):>9.4f}{_m('macroAUPRC'):>8.4f}"
              f"{_m('true_causal_pct'):>7.1f}{_m('fallback_pct'):>6.1f}{_m('abstain_pct'):>7.1f}"
              f"{_m('prior_overlap_pct'):>8.1f}{_m('leak_pct'):>7.1f}{_m('band_occ_pct'):>7.1f}{_m('mean_sim'):>6.3f}")
    print("=" * 118)
    print("An arm that cannot beat CTRL:* AUROC is not causal signal. H/I isolate the v14 scoring fixes.")

    if PAIRED_ROWS:
        dfp = pd.DataFrame(PAIRED_ROWS)
        print("\n" + "=" * 78 + "\nPAIRED dAUROC vs FULL  (mean over seeds; * = CI excludes 0)\n" + "=" * 78)
        print(f"{'config':<20}{'d_macroAUROC':>14}{'95% CI':>24}{'p(<=0)':>10}")
        for lab in [l for l, _ in CONFIGS if l != 'FULL']:
            s = dfp[dfp['label'] == lab]
            if s.empty: continue
            dm = float(np.nanmean(s['d_macroAUROC'])); lo = float(np.nanmean(s['d_lo'])); hi = float(np.nanmean(s['d_hi']))
            p = float(np.nanmean(s['p_delta_le0'])); star = " *" if (lo > 0 or hi < 0) else ""
            print(f"{lab:<20}{dm:>+14.4f}{('['+format(lo,'+.4f')+','+format(hi,'+.4f')+']'):>24}{p:>10.3f}{star}")

    if full_pc:
        print("\n" + "=" * 104 + "\nFULL PER-CLASS BENCHMARK  [95% CI]  vs CONTROLS\n" + "=" * 104)
        ctrl_pc = {r["class"]: r for r in PC_ROWS
                   if str(r["label"]) == "CTRL:random" and int(r["seed"]) == ABL_SEEDS[0]}
        print(f"{'class':<16}{'AUROC':>8}{'95% CI':>18}{'AUPRC':>8}{'rand.AUROC':>11}{'pos/neg':>10}{'abst%':>7}")
        for r in full_pc:
            ci = f"[{r['ci_lo']:.3f},{r['ci_hi']:.3f}]" if r['ci_lo'] == r['ci_lo'] else "  n/a"
            cr = ctrl_pc.get(r["class"], {})
            print(f"{r['class']:<16}{r['auroc']:>8.3f}{ci:>18}{r['auprc']:>8.3f}"
                  f"{cr.get('auroc', float('nan')):>11.3f}"
                  f"{str(r['support_pos'])+'/'+str(r['support_neg']):>10}{r.get('abstain_pct', float('nan')):>7.1f}")

    dfR_full = dfR[dfR["seed"] == ABL_SEEDS[0]]
    labels = [l for l in _ORDER if l in set(dfR_full["label"].astype(str))]
    def _rowval(lab, col):
        s = dfR_full[dfR_full["label"].astype(str) == lab]
        return float(pd.to_numeric(s[col], errors="coerce").mean()) if not s.empty else float("nan")
    auroc_max = [_rowval(l, "macroAUROC") for l in labels]; auprc_v = [_rowval(l, "macroAUPRC") for l in labels]
    ctrl_rand = _rowval("CTRL:random", "macroAUROC"); x = np.arange(len(labels))
    fig, ax1 = plt.subplots(figsize=(16, 6))
    ax1.bar(x - 0.2, auroc_max, 0.4, label="macroAUROC (max)", color="#2ca02c")
    ax1.bar(x + 0.2, auprc_v, 0.4, label="macroAUPRC", color="#1f77b4")
    if ctrl_rand == ctrl_rand:
        ax1.axhline(ctrl_rand, color="#d62728", ls="--", lw=1.5, label=f"random-patch AUROC={ctrl_rand:.3f}")
    ax1.axhline(0.5, color="#888", ls=":", lw=1); ax1.set_ylabel("score"); ax1.set_ylim(0, 1.0)
    ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=55, ha="right", fontsize=8)
    ax1.legend(loc="upper right", fontsize=9)
    ax1.set_title(f"Patch-mining ablation (seed {ABL_SEEDS[0]}, n={ABL_SUBSET_N}) - arms vs negative control")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/ablation_overall.png", dpi=110, bbox_inches="tight"); plt.show()

    _all_done = all(f"{lab}|{s}" in done for lab in _ORDER for s in ABL_SEEDS)
    print(f"\n{'Ablations COMPLETE' if _all_done else 'PARTIAL - re-run to finish'}")
    print(f"   overall -> {ABL_OUT_CSV}\n   perclass -> {ABL_PERCLASS_CSV}\n   paired -> {ABL_PAIRED_CSV}")
